# REHAB24-6: 3D World Landmarks & Calibration Benchmark (`chamodi-week2`)

Validates **Variant B** (`angles_world.py`) and **Variant D** (`calibration.py`) against
REHAB24-6 OptiTrack 3D ground-truth data.

**Run cells 1 → 6 in order. No changes needed.**

In [1]:
# ============================================================
# STEP 1 — Python 3.12 environment for MediaPipe
# ============================================================
import os, subprocess

MP_PY = '/content/mpenv/bin/python'
if not os.path.exists(MP_PY):
    print('Installing uv and creating Python 3.12 env...')
    subprocess.run(['pip', 'install', '-q', 'uv'], check=True)
    subprocess.run(['uv', 'venv', '--python', '3.12', '/content/mpenv'], check=True)
    subprocess.run([
        'uv', 'pip', 'install', '--python', MP_PY,
        'numpy==1.26.4', 'mediapipe==0.10.14', 'pyyaml',
        'pandas', 'opencv-python', 'matplotlib',
        'scikit-learn', 'scipy',
    ], check=True)
print('MediaPipe Python 3.12 env ready at:', MP_PY)

Installing uv and creating Python 3.12 env...
MediaPipe Python 3.12 env ready at: /content/mpenv/bin/python


In [33]:
# ============================================================
# FULL RESTART CELL — run this after any kernel disconnect
# ============================================================
import glob, os, shutil, subprocess, sys, time, zipfile
import numpy as np, pandas as pd

# reset working directory FIRST — prevents "No such file or directory" error
os.chdir('/content')

# ---- core settings ----
REPO_DIR         = '/content/repo'
BRANCH           = 'chamodi-week2'
REPO_URL         = 'https://github.com/suvini2001/AI-based-Physiotherapy-project-.git'
MP_PY            = '/content/mpenv/bin/python'
DATA             = '/content/rehab24'
MAX_VIDEOS       = None
EXERCISES        = [1, 2, 3]
THRESHOLD        = 0.70
MARGIN           = 45
MAX_LAG          = 30
MODEL_COMPLEXITY = 1

ARMS_OVERRIDE = {
    'right': ('12: RightArm', '13: RightForeArm', '14: RightHand'),
    'left':  ('7: LeftArm',   '8: LeftForeArm',   '9: LeftHand'),
}

# ---- mount Drive ----
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
CACHE   = '/content/drive/MyDrive/rehab24_cache_3d'
RESULTS = '/content/drive/MyDrive/rehab24_results_3d'
for d in (DATA, CACHE, RESULTS):
    os.makedirs(d, exist_ok=True)

# ---- remove any old clone and clone correct branch ----
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
assert not os.path.exists(REPO_DIR), "Could not remove old repo"
subprocess.run(
    ['git', 'clone', '-q', '-b', BRANCH, REPO_URL, REPO_DIR],
    check=True, cwd='/content'
)
os.chdir(REPO_DIR)
for p in (REPO_DIR,
          os.path.join(REPO_DIR, 'capture'),
          os.path.join(REPO_DIR, 'pose'),
          os.path.join(REPO_DIR, 'vision')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('Branch:', BRANCH)

# ---- verify modules ----
import importlib.util
all_ok = True
for mod in ['angles', 'angles_world', 'benchmark_common',
            'calibration', 'filters', 'visibility']:
    spec = importlib.util.find_spec(mod)
    status = 'OK' if spec else 'MISSING!!!'
    if not spec: all_ok = False
    print(f'  {mod:20s}: {status}')
assert all_ok, "Some modules missing — check branch"

# ---- joint names ----
jn_files = glob.glob(f'{DATA}/**/*joint*names*.txt', recursive=True)
assert jn_files, "joints_names.txt not found — re-run Step 3 first"
with open(jn_files[0]) as f:
    JOINT_NAMES = [l.strip() for l in f if l.strip()]
print(f'Joint names: {len(JOINT_NAMES)} loaded')

for side, joints in ARMS_OVERRIDE.items():
    for jn in joints:
        assert jn in JOINT_NAMES, f"'{jn}' not in JOINT_NAMES"
print('ARMS_OVERRIDE: all joints verified OK')

# ---- segmentation ----
seg_files = glob.glob(f'{DATA}/**/Segmentation.*', recursive=True)
assert seg_files, "Segmentation.csv not found — re-run Step 3 first"
sep = ';' if ';' in open(seg_files[0]).readline() else ','
seg_df = pd.read_csv(seg_files[0], sep=sep)
cols_lower = {c.lower(): c for c in seg_df.columns}

def pick_col(candidates):
    for cand in candidates:
        for cl, orig in cols_lower.items():
            if cand in cl: return orig
    raise KeyError(f'No column matching {candidates}')

c_vid   = pick_col(['video_id', 'video', 'file', 'rec', 'name'])
c_ex    = pick_col(['exercise_id', 'exercise', 'activity', 'ex'])
c_start = pick_col(['first_frame', 'start', 'begin', 'f0', 'from'])
c_end   = pick_col(['last_frame',  'end',   'stop',  'f1', 'to'])
c_view  = None
for cl, orig in cols_lower.items():
    if any(k in cl for k in ['cam17_orientation', 'view', 'angle', 'cam_pos']):
        c_view = orig
        break
print(f'Columns: vid={c_vid}, ex={c_ex}, start={c_start}, end={c_end}, view={c_view}')

# ---- index files ----
all_videos = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob(f'{DATA}/**/*.*', recursive=True)
    if p.lower().endswith(('.mp4', '.avi', '.mov'))
}
SKIP = {'joints_names.txt', 'joint_names.txt',
        'segmentation.csv', 'segmentation.txt'}
all_3d = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob(f'{DATA}/**/*.*', recursive=True)
    if p.lower().endswith(('.npy', '.npz'))
    and os.path.basename(p).lower() not in SKIP
}
print(f'Videos: {len(all_videos)},  3D truth files: {len(all_3d)}')

# ---- build recordings ----
seg_ex = seg_df[seg_df[c_ex].isin(EXERCISES)].copy()
recordings = []
for (vid_id, ex_id), grp in seg_ex.groupby([c_vid, c_ex]):
    person_id = str(vid_id).strip()
    f0 = max(0, int(grp[c_start].min()) - MARGIN)
    f1 = int(grp[c_end].max()) + MARGIN
    view_val = str(grp[c_view].iloc[0]) if c_view else 'unknown'
    matched_vids = {k: v for k, v in all_videos.items()
                    if k.startswith(person_id + '-') or k == person_id}
    person_3d = {k: v for k, v in all_3d.items()
                 if k.startswith(person_id + '-') or k == person_id}
    if not person_3d: continue
    truth_key = next((k for k in person_3d if '30fps' in k),
                    next((k for k in person_3d if '120fps' in k), next(iter(person_3d))))
    for vid_key, vid_path in matched_vids.items():
        cam = ('cam18'  if 'cam18' in vid_key.lower() or 'transposed' in vid_key.lower()
               else 'cam17' if 'cam17' in vid_key.lower()
               else 'unknown')
        recordings.append({
            'rec_id':     f'{person_id}_ex{int(ex_id)}_{cam}',
            'video_key':  vid_key,
            'video_path': vid_path,
            'truth_path': person_3d[truth_key],
            'truth_key':  truth_key,
            'exercise':   int(ex_id),
            'f0': f0, 'f1': f1,
            'view_raw':   view_val,
            'cam':        cam,
            'cam_label':  ('cam18' if 'camera18' in vid_key.lower() or 'cam18' in vid_key.lower()
                           else 'cam17' if 'camera17' in vid_key.lower() or 'cam17' in vid_key.lower()
                           else 'unknown'),
        })

recordings = list({r['rec_id']: r for r in recordings}.values())
if MAX_VIDEOS:
    recordings = recordings[:MAX_VIDEOS]
print(f'Total recordings: {len(recordings)}')

# ---- cache status ----
cached_files = {os.path.basename(p) for p in glob.glob(f'{CACHE}/*.npz')}
matched = sum(1 for r in recordings
              if f"{r['rec_id']}_{r['f0']}_{r['f1']}_3d.npz" in cached_files)
print(f'Cache: {matched}/{len(recordings)} done — {len(recordings)-matched} remaining')
print('\nReady — now run Step 5')

Mounted at /content/drive
Branch: chamodi-week2
  angles              : OK
  angles_world        : OK
  benchmark_common    : OK
  calibration         : OK
  filters             : OK
  visibility          : OK
Joint names: 26 loaded
ARMS_OVERRIDE: all joints verified OK
Columns: vid=video_id, ex=exercise_id, start=first_frame, end=last_frame, view=cam17_orientation
Videos: 130,  3D truth files: 130
Total recordings: 70
Cache: 70/70 done — 0 remaining

Ready — now run Step 5


In [3]:
# ============================================================
# STEP 3 — Locate dataset in Google Drive
# ============================================================
import urllib.request

ZENODO_BASE = 'https://zenodo.org/records/13305826/files'

def find_in_drive(filename):
    if not os.path.exists('/content/drive/MyDrive'):
        return None
    matches = glob.glob(f'/content/drive/MyDrive/**/{filename}', recursive=True)
    return matches[0] if matches else None

def ensure_file(filename, url_name=None):
    target = os.path.join(DATA, filename)
    if os.path.exists(target) and os.path.getsize(target) > 0:
        print(f'[OK]       {filename}')
        return target
    hit = find_in_drive(filename)
    if hit:
        print(f'[Drive]    {filename} <- {hit}')
        if not os.path.exists(target):
            os.symlink(hit, target)
        return target
    url = f"{ZENODO_BASE}/{url_name or filename}?download=1"
    print(f'[Download] {url}')
    urllib.request.urlretrieve(url, target)
    return target

# --- Segmentation.csv ---
for name in ('Segmentation.csv', 'Segmentation.txt'):
    hit = find_in_drive(name)
    if hit:
        dst = os.path.join(DATA, os.path.basename(hit))
        if not os.path.exists(dst):
            shutil.copy(hit, dst)
        break
if not glob.glob(f'{DATA}/Segmentation.*'):
    try: ensure_file('Segmentation.csv')
    except: ensure_file('Segmentation.txt')

# --- joints_names.txt ---
for name in ('joints_names.txt', 'joint_names.txt'):
    hit = find_in_drive(name)
    if hit:
        dst = os.path.join(DATA, os.path.basename(hit))
        if not os.path.exists(dst):
            shutil.copy(hit, dst)
        break
if not glob.glob(f'{DATA}/joint*names*.txt'):
    ensure_file('joints_names.txt')

# --- videos (Ex1..Ex6 structure OR videos.zip) ---
# First check if videos already extracted into Ex* folders
existing_mp4s = glob.glob(f'{DATA}/**/*.mp4', recursive=True) + \
                glob.glob(f'{DATA}/**/*.avi', recursive=True)
if not existing_mp4s:
    # try Drive zip
    drive_zip = find_in_drive('videos.zip')
    if drive_zip:
        target = os.path.join(DATA, 'videos.zip')
        if not os.path.exists(target):
            os.symlink(drive_zip, target)
        print('Extracting videos.zip...')
        with zipfile.ZipFile(target) as zf:
            zf.extractall(DATA)
    else:
        z = ensure_file('videos.zip')
        print('Extracting videos.zip...')
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATA)

# --- 3d_joints ---
existing_3d = glob.glob(f'{DATA}/3d_joints/**/*.*', recursive=True)
if not existing_3d:
    drive_3d_dir = None
    hits = glob.glob('/content/drive/MyDrive/**/3d_joints', recursive=True)
    if hits and os.path.isdir(hits[0]) and os.listdir(hits[0]):
        drive_3d_dir = hits[0]
    if drive_3d_dir:
        dst = os.path.join(DATA, '3d_joints')
        if not os.path.exists(dst):
            os.symlink(drive_3d_dir, dst)
        print(f'[Drive] Linked 3d_joints from {drive_3d_dir}')
    else:
        z = ensure_file('3d_joints.zip')
        print('Extracting 3d_joints.zip...')
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATA)

print('\nDATA contents:', os.listdir(DATA))
all_mp4s = glob.glob(f'{DATA}/**/*.mp4', recursive=True) + \
           glob.glob(f'{DATA}/**/*.avi', recursive=True)
print(f'Total video files found: {len(all_mp4s)}')
if all_mp4s:
    print('  Example:', all_mp4s[0])

Extracting videos.zip...
[Download] https://zenodo.org/records/13305826/files/3d_joints.zip?download=1
Extracting 3d_joints.zip...

DATA contents: ['joints_names.txt', 'Ex2', 'Ex4', 'Ex6', 'Ex1', '3d_joints.zip', 'Ex3', 'Ex5', 'Segmentation.csv', 'videos.zip']
Total video files found: 130
  Example: /content/rehab24/Ex2/PM_110-Camera18-30fps-transposed.mp4


In [4]:
# ============================================================
# STEP 4 — Parse segmentation + build recordings list
# ============================================================
import numpy as np
import pandas as pd

# --- joint names ---
jn_files = glob.glob(f'{DATA}/**/*joint*names*.txt', recursive=True)
assert jn_files, 'joints_names.txt not found'
with open(jn_files[0]) as f:
    JOINT_NAMES = [l.strip() for l in f if l.strip()]
print(f'Loaded {len(JOINT_NAMES)} OptiTrack joint names')
print('  First 8:', JOINT_NAMES[:8])

# verify the arm joint names we need are present
for side, (sh, el, wr) in ARMS_OVERRIDE.items():
    for jn in (sh, el, wr):
        if jn not in JOINT_NAMES:
            print(f'  WARNING: {jn} not in JOINT_NAMES — check ARMS_OVERRIDE')
        else:
            print(f'  OK: {jn} found (idx {JOINT_NAMES.index(jn)})')

# --- segmentation ---
seg_files = glob.glob(f'{DATA}/**/Segmentation.*', recursive=True)
assert seg_files, 'Segmentation.csv not found'
sep = ';' if ';' in open(seg_files[0]).readline() else ','
seg_df = pd.read_csv(seg_files[0], sep=sep)
print('\nSegmentation columns:', list(seg_df.columns))
print(f'Total rows: {len(seg_df)}')

cols_lower = {c.lower(): c for c in seg_df.columns}

def pick_col(candidates):
    for cand in candidates:
        for cl, orig in cols_lower.items():
            if cand in cl:
                return orig
    raise KeyError(f'No column matching {candidates} in {list(seg_df.columns)}')

c_vid   = pick_col(['video_id', 'video', 'file', 'rec', 'name'])
c_ex    = pick_col(['exercise_id', 'exercise', 'activity', 'ex'])
c_start = pick_col(['first_frame', 'start', 'begin', 'f0', 'from'])
c_end   = pick_col(['last_frame', 'end', 'stop', 'f1', 'to'])

c_view = None
for cl, orig in cols_lower.items():
    if any(k in cl for k in ['cam17_orientation', 'view', 'angle', 'cam_pos']):
        c_view = orig
        break

print(f'Using columns: video={c_vid}, exercise={c_ex}, start={c_start}, end={c_end}, view={c_view}')

# --- index ALL video files (search whole DATA tree, not just videos/) ---
all_videos = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob(f'{DATA}/**/*.*', recursive=True)
    if p.lower().endswith(('.mp4', '.avi', '.mov'))
}

# --- index 3D ground-truth files ---
all_3d = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob(f'{DATA}/**/*.*', recursive=True)
    if p.lower().endswith(('.npy', '.npz', '.txt', '.csv'))
    and os.path.basename(p).lower() not in
       ('joints_names.txt', 'joint_names.txt', 'segmentation.csv', 'segmentation.txt')
    and '3d_joints' in p
}

print(f'\nIndexed {len(all_videos)} video files')
print(f'Indexed {len(all_3d)} 3D ground-truth files')
if all_videos:
    print('  Video example:', list(all_videos.keys())[0])
if all_3d:
    print('  3D truth example:', list(all_3d.keys())[0])

# --- build recordings ---
seg_ex = seg_df[seg_df[c_ex].isin(EXERCISES)].copy()
recordings = []

for (vid_id, ex_id), grp in seg_ex.groupby([c_vid, c_ex]):
    vid_str  = str(vid_id).strip()
    base_id  = vid_str.split('-')[0]   # 'PM_013' from 'PM_013-cam17'
    f0       = max(0, int(grp[c_start].min()) - MARGIN)
    f1       = int(grp[c_end].max()) + MARGIN
    view_val = str(grp[c_view].iloc[0]) if c_view else 'unknown'

    # match video files by exact name or prefix
    matched_vids = [
        k for k in all_videos
        if k == vid_str
        or k.startswith(vid_str + '-')
        or k.startswith(base_id + '-')
    ]

    # match 3D truth by person ID (PM_013)
    truth_key = next((k for k in all_3d if base_id in k), None)

    if not truth_key:
        continue

    for mv in sorted(set(matched_vids)):
        recordings.append({
            'rec_id':     f'{mv}_ex{int(ex_id)}',
            'video_key':  mv,
            'video_path': all_videos[mv],
            'truth_path': all_3d[truth_key],
            'truth_key':  truth_key,
            'exercise':   int(ex_id),
            'f0': f0, 'f1': f1,
            'view_raw': view_val,
        })

# deduplicate
recordings = list({r['rec_id']: r for r in recordings}.values())
if MAX_VIDEOS is not None:
    recordings = recordings[:MAX_VIDEOS]

print(f'\nPrepared {len(recordings)} recordings for evaluation')
if not recordings:
    print('\n  !! 0 recordings found. Debug info:')
    print('  seg_df video_id sample:', seg_df[c_vid].unique()[:5])
    print('  all_videos keys sample:', list(all_videos.keys())[:5])
    print('  all_3d keys sample:    ', list(all_3d.keys())[:5])
else:
    for r in recordings[:3]:
        print(f"  {r['rec_id']}  view={r['view_raw']}")

Loaded 26 OptiTrack joint names
  First 8: ['0: Hips', '1: Spine', '2: Spine1', '3: Neck', '4: Head', '5: Head_end', '6: LeftShoulder', '7: LeftArm']
  OK: 12: RightArm found (idx 12)
  OK: 13: RightForeArm found (idx 13)
  OK: 14: RightHand found (idx 14)
  OK: 7: LeftArm found (idx 7)
  OK: 8: LeftForeArm found (idx 8)
  OK: 9: LeftHand found (idx 9)

Segmentation columns: ['video_id', 'repetition_number', 'exercise_id', 'person_id', 'first_frame', 'last_frame', 'cam17_orientation', 'mocap_erroneous', 'exercise_subtype', 'lights_on', 'extra_person_in_cam17', 'extra_person_in_cam18', 'correctness']
Total rows: 1072
Using columns: video=video_id, exercise=exercise_id, start=first_frame, end=last_frame, view=cam17_orientation

Indexed 130 video files
Indexed 0 3D ground-truth files
  Video example: PM_110-Camera18-30fps-transposed

Prepared 0 recordings for evaluation

  !! 0 recordings found. Debug info:
  seg_df video_id sample: ['PM_000' 'PM_001' 'PM_002' 'PM_012' 'PM_016']
  all_vid

In [ ]:
import glob
# see what actually got extracted
all_npy = glob.glob('/content/rehab24/**/*.npy', recursive=True)
print(f"Total .npy files found: {len(all_npy)}")
if all_npy:
    for p in all_npy[:5]:
        print(" ", p)
else:
    print("No .npy files anywhere in /content/rehab24")
    print("Contents of /content/rehab24:", os.listdir('/content/rehab24'))
    # check if 3d_joints folder exists
    import os
    for root, dirs, files in os.walk('/content/rehab24'):
        depth = root.replace('/content/rehab24','').count('/')
        if depth <= 2:
            print(' '*depth*2 + os.path.basename(root) + '/')
            for f in files[:3]:
                print(' '*(depth*2+2) + f)

In [5]:
import glob, os

SKIP = {'joints_names.txt', 'joint_names.txt',
        'segmentation.csv', 'segmentation.txt'}

all_3d = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob('/content/rehab24/**/*.npy', recursive=True)
    if os.path.basename(p).lower() not in SKIP
}

print(f"all_3d: {len(all_3d)} files")
for k in list(all_3d.keys())[:5]:
    print(f"  '{k}' -> {all_3d[k]}")

all_3d: 130 files
  'PM_102-30fps' -> /content/rehab24/Ex2/PM_102-30fps.npy
  'PM_013-120fps' -> /content/rehab24/Ex2/PM_013-120fps.npy
  'PM_110-30fps' -> /content/rehab24/Ex2/PM_110-30fps.npy
  'PM_025-120fps' -> /content/rehab24/Ex2/PM_025-120fps.npy
  'PM_115-30fps' -> /content/rehab24/Ex2/PM_115-30fps.npy


In [6]:
print("recordings:", len(recordings))
print("JOINT_NAMES:", len(JOINT_NAMES))
print("all_videos:", len(all_videos))
print("all_3d:", len(all_3d))

recordings: 0
JOINT_NAMES: 26
all_videos: 130
all_3d: 130


In [34]:
seg_ex = seg_df[seg_df[c_ex].isin(EXERCISES)].copy()
recordings = []
for (vid_id, ex_id), grp in seg_ex.groupby([c_vid, c_ex]):
    person_id = str(vid_id).strip()
    f0 = max(0, int(grp[c_start].min()) - MARGIN)
    f1 = int(grp[c_end].max()) + MARGIN
    view_val = str(grp[c_view].iloc[0]) if c_view else 'unknown'
    matched_vids = {k: v for k, v in all_videos.items()
                    if k.startswith(person_id + '-') or k == person_id}
    person_3d = {k: v for k, v in all_3d.items()
                 if k.startswith(person_id + '-') or k == person_id}
    if not person_3d: continue
    truth_key = next((k for k in person_3d if '30fps' in k),
                     next((k for k in person_3d if '120fps' in k), next(iter(person_3d))))
    for vid_key, vid_path in matched_vids.items():
        cam = ('cam18'  if 'cam18' in vid_key.lower() or 'transposed' in vid_key.lower()
               else 'cam17' if 'cam17' in vid_key.lower()
               else 'unknown')
        recordings.append({
            'rec_id':     f'{person_id}_ex{int(ex_id)}_{cam}',
            'video_key':  vid_key,
            'video_path': vid_path,
            'truth_path': person_3d[truth_key],
            'truth_key':  truth_key,
            'exercise':   int(ex_id),
            'f0': f0, 'f1': f1,
            'view_raw':   view_val,
            'cam':        cam,
            'cam_label':  ('cam18' if 'camera18' in vid_key.lower() or 'cam18' in vid_key.lower()
                           else 'cam17' if 'camera17' in vid_key.lower() or 'cam17' in vid_key.lower()
                           else 'unknown'),

        })

recordings = list({r['rec_id']: r for r in recordings}.values())
print(f'Recordings rebuilt: {len(recordings)}')

cached_files = {os.path.basename(p) for p in glob.glob(f'{CACHE}/*.npz')}
matched = sum(1 for r in recordings
              if f"{r['rec_id']}_{r['f0']}_{r['f1']}_3d.npz" in cached_files)
print(f'Cache: {matched}/{len(recordings)} done — {len(recordings)-matched} remaining')

Recordings rebuilt: 70
Cache: 70/70 done — 0 remaining


In [36]:
# ============================================================
# STEP 5 (INCREMENTAL SAVE) — saves after every recording
# ============================================================
import numpy as np, os, subprocess, time, csv

import angles as angles_2d
import pandas as pd
from angles_world import elbow_angle_3d, mask_world_like
import benchmark_common as bc
from benchmark_common import elbow_angle_3d_ordered
from calibration import fit_curve, apply_calibration
from benchmark_common import butterworth_filter
from visibility import mask_low_visibility
from calibration import fit_curve, apply_calibration, leave_one_person_out

def fill_gaps_cubic(arr, max_gap=5):
    s = pd.Series(arr, dtype=float)
    # Cubic spline සඳහා අවම වශයෙන් වලංගු ලක්ෂ්‍ය 4ක් තිබිය යුතුය
    if s.notna().sum() >= 4:
        filled = s.interpolate(method='cubicspline', limit=max_gap, limit_area='inside')
    else:
        filled = s.interpolate(method='linear', limit=max_gap, limit_area='inside')
    # කෝණය 0° සහ 180° අතර තබාගැනීමට clip කරන්න
    return np.clip(filled.to_numpy(), 0.0, 180.0)


print('All imports OK')

def fill_1d(arr, max_gap=15):
    arr = np.array(arr, dtype=float)
    if np.isnan(arr).all() or not np.isnan(arr).any():
        return arr
    idx  = np.arange(len(arr))
    good = ~np.isnan(arr)
    filled = arr.copy()
    in_gap, gap_start = False, 0
    for i in range(len(arr)):
        if np.isnan(arr[i]) and not in_gap:
            in_gap, gap_start = True, i
        elif not np.isnan(arr[i]) and in_gap:
            in_gap = False
            if i - gap_start <= max_gap:
                filled[gap_start:i] = np.interp(
                    idx[gap_start:i], idx[good], arr[good])
    return filled

def infer_view(rec):
    """cam17_orientation describes the SUBJECT's orientation w.r.t. camera 17.
    It must not be reused as camera 18's viewpoint."""
    v   = str(rec.get('view_raw', '')).lower()
    cam = rec.get('cam_label', rec.get('cam', ''))

    if 'half' in v:                   subj = 'Half-profile'
    elif 'front' in v or v == '0':    subj = 'Front'
    elif 'prof' in v:                 subj = 'Profile'
    elif 'side' in v or '90' in v:    subj = 'Side'
    else:                             subj = 'Unknown'

    if cam == 'cam18':
        return f'{subj}/cam18'        # same subject pose, different camera
    return subj

def get_3d_landmarks(rec):
    cache_path = os.path.join(
        CACHE, f"{rec['rec_id']}_{rec['f0']}_{rec['f1']}_3d.npz")
    need_extract = True
    if os.path.exists(cache_path):
        try:
            d = np.load(cache_path)
            if 'world' in d.files and 'lm' in d.files and d['lm'].shape[0] > 0:
                need_extract = False
        except Exception:
            pass
    if need_extract:
        env = os.environ.copy()
        env['MPLBACKEND'] = 'Agg'
        env['PYTHONPATH'] = REPO_DIR
        cmd = [MP_PY, 'extract_world.py',
               rec['video_path'],
               str(rec['f0']), str(rec['f1']),
               cache_path, str(MODEL_COMPLEXITY)]
        subprocess.run(cmd, check=True, env=env, cwd=REPO_DIR)
    d = np.load(cache_path)
    return d['lm'], d['world'], tuple(d['size']), float(d['fps'])

# ---- incremental CSV writers ----
path_A = os.path.join(RESULTS, 'variant_A_2d.csv')
path_B = os.path.join(RESULTS, 'variant_B_3d_world.csv')
path_D = os.path.join(RESULTS, 'variant_D_3d_calibrated.csv')

# check how many already saved (resume support)
def append_row(path, row):
    """Append one row dict to a CSV, creating header if file is new/empty."""
    write_header = not os.path.exists(path) or os.path.getsize(path) == 0
    with open(path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if write_header:
            writer.writeheader()
        writer.writerow(row)

# fresh start: the old rows came from the buggy run, and append_row only appends
for path in (path_A, path_B, path_D):
    if os.path.exists(path):
        os.remove(path)
already_done = set()      # no resume; a full re-run is cheap (landmarks are cached)


def resample_to(t_target, t_src, values, max_gap_s=0.1):
    """Linear-interpolate 1-D values onto t_target. NaN-safe: a real mocap
    dropout stays NaN instead of being interpolated across."""
    values = np.asarray(values, float)
    good = np.isfinite(values)
    if good.sum() < 2:
        return np.full(len(t_target), np.nan)
    out = np.interp(t_target, t_src[good], values[good], left=np.nan, right=np.nan)
    if max_gap_s is not None:
        idx = np.searchsorted(t_src[good], t_target)
        idx = np.clip(idx, 1, good.sum() - 1)
        near = np.minimum(np.abs(t_target - t_src[good][idx]),
                          np.abs(t_target - t_src[good][idx - 1]))
        out[near > max_gap_s] = np.nan
    return out

def truth_angle_for(rec, truth_arr, world, npz_f0, fps, arm_joints):
    """gt angle on the VIDEO frame grid, whatever rate the truth file is."""
    TRUTH_FPS = 120.0 if '120fps' in str(rec.get('truth_key', '')) else 30.0
    if abs(TRUTH_FPS - fps) < 1.0:
        win = truth_arr[npz_f0: npz_f0 + len(world)]
        return bc.truth_elbow_3d(win, JOINT_NAMES, arm_joints, use_dims=3)
    gt_full = bc.truth_elbow_3d(truth_arr, JOINT_NAMES, arm_joints, use_dims=3)
    t_truth = np.arange(len(truth_arr)) / TRUTH_FPS
    t_video = (npz_f0 + np.arange(len(world))) / fps
    return resample_to(t_video, t_truth, gt_full)


skipped_align = 0
skipped_static = 0
skipped_cov = 0
MIN_ROM = 60.0        # Ex1 max 50.8 deg, Ex2/Ex3 min 80.4 deg -> clean split
MIN_COV = 50.0
rows_A, rows_B, aligned_cache = [], [], []

for idx, rec in enumerate(recordings, 1):
    try:
        t0 = time.time()
        lm, world, size, fps = get_3d_landmarks(rec)
        truth_arr = bc.load_truth_array(rec['truth_path'], JOINT_NAMES)

        npz_f0    = int(np.load(os.path.join(
            CACHE, f"{rec['rec_id']}_{rec['f0']}_{rec['f1']}_3d.npz"))['f0'])
        TRUTH_FPS = 120.0 if "120fps" in str(rec.get("truth_key", "")) else 30.0
        if abs(TRUTH_FPS - fps) < 1.0:
            truth_win = truth_arr[npz_f0: npz_f0 + len(world)]   # rates match
            t_video = t_truth = None
        else:
            truth_win = None                                      # interp needed
            t_truth = np.arange(len(truth_arr)) / TRUTH_FPS
            t_video = (npz_f0 + np.arange(len(world))) / fps

        lm_masked    = mask_low_visibility(lm, threshold=THRESHOLD)
        world_masked = mask_world_like(world, lm_masked)

        side_mae = {'right': float('nan'), 'left': float('nan')}

        for side in ('right', 'left'):
            arm_joints = ARMS_OVERRIDE[side]
            if truth_win is not None:
                gt_3d = bc.truth_elbow_3d(truth_win, JOINT_NAMES, arm_joints, use_dims=3)
            else:
                gt_full = bc.truth_elbow_3d(truth_arr, JOINT_NAMES, arm_joints, use_dims=3)
                gt_3d = resample_to(t_video, t_truth, gt_full)

            pred_b_raw = elbow_angle_3d_ordered(world_masked, side=side)
            pred_b_fill = fill_1d(pred_b_raw, max_gap=5)
            pred_b = (butterworth_filter(pred_b_fill, fs=fps, cutoff=6.0)
                      if np.isfinite(pred_b_fill).sum() > 15 else pred_b_fill)

            pred_a_raw  = angles_2d.elbow_angle(lm_masked, side=side, size=size)
            pred_a_fill = fill_1d(pred_a_raw, max_gap=5)
            pred_a = (butterworth_filter(pred_a_fill, fs=fps, cutoff=6.0)
                      if np.isfinite(pred_a_fill).sum() > 15 else pred_a_fill)

            meta = {
                'recording': f"{rec['rec_id']}_{side}",
                'video':     rec['video_key'],
                'person':    bc.person_of(rec['video_key']),
                'ex':        rec['exercise'],
                'cam':       rec.get('cam_label', rec.get('cam', 'unknown')),
                'side':      side,
                'view':      infer_view(rec),
            }

                        # --- gate 1: does this joint actually flex? (paper p.10) ---
            gt_range = (float(np.nanpercentile(gt_3d, 95) - np.nanpercentile(gt_3d, 5))
                        if np.isfinite(gt_3d).any() else float('nan'))
            meta = dict(meta, gt_range=round(gt_range, 1) if gt_range == gt_range else None)

            if not (gt_range >= MIN_ROM):
                skipped_static += 1
                print(f"    static {side}: ROM {gt_range:.1f}deg < {MIN_ROM:.0f} - no expected flexion")
                continue
                        # --- gate 1b: is this arm visible at all? (far arm in profile views) ---
            cov_a = 100.0 * float(np.isfinite(pred_a).mean())
            meta = dict(meta, cov_pred=round(cov_a, 1))

            if cov_a < MIN_COV:
                skipped_cov += 1
                print(f"    occluded {side}: coverage {cov_a:.1f}% "
                      f"(ROM {gt_range:.0f}deg is fine) - arm hidden from this camera")
                continue

            # --- gate 2: temporal alignment ---
            lag, lag_r = bc.best_lag(pred_a, gt_3d, max_lag=MAX_LAG, filter_signals=True)
            pa_al, gt_al = bc.overlap(pred_a, gt_3d, lag)
            pb_al, _     = bc.overlap(pred_b, gt_3d, lag)
            meta = dict(meta, lag=lag, lag_r=round(float(lag_r), 3))

            if abs(lag) >= MAX_LAG or lag_r < 0.7:
                skipped_align += 1
                print(f"Skipped (poor tracking, r < 0.7)      : {skipped_align}")
                continue


            row_a = bc.make_row("A_2D_Base",  pa_al, gt_al, meta)
            row_b = bc.make_row("B_3D_World", pb_al, gt_al, meta)

            if row_b:
                rows_B.append(row_b)
                append_row(path_B, row_b)
                side_mae[side] = row_b["mae"]        # track for print
            if row_a:
                rows_A.append(row_a)
                append_row(path_A, row_a)
                aligned_cache.append({"meta": meta, "pred_2d": pa_al,
                                      "pred_3d": pb_al, "truth_3d": gt_al})

        elapsed = time.time() - t0
        mae_r, mae_l = side_mae['right'], side_mae['left']
        print(f'[{idx}/{len(recordings)}] {rec["rec_id"]}  '
              f'{elapsed:.1f}s  MAE R/L: {mae_r:.1f}°/{mae_l:.1f}°')

    except Exception as e:
        import traceback
        print(f'[{idx}/{len(recordings)}] SKIPPED {rec["rec_id"]}: {e}')
        traceback.print_exc()


print(f"\nSkipped (no flexion, ROM < {MIN_ROM:.0f}deg)  : {skipped_static}")
print(f"Skipped (occluded, cov < {MIN_COV:.0f}%)       : {skipped_cov}")
print(f"Skipped (misaligned, lag/r)           : {skipped_align}")
print(f"Kept                                  : {len(rows_A)}")

# ---- Variant D: LOPO calibration — global and per-view ----
rows_D = []
for item in aligned_cache:
    me = item['meta']
    pool = [x for x in aligned_cache if x['meta']['person'] != me['person']]
    if not pool:
        continue
    same_view = [x for x in pool if x['meta']['view'] == me['view']]

    jobs = [('pred_2d', pool,      'D_2D_Calibrated'),
            ('pred_3d', pool,      'D_3D_Calibrated'),
            ('pred_2d', same_view, 'D_2D_PerView')]

    for key, train, label in jobs:
        if label.endswith('PerView') and len(train) < 3:
            train = pool                      # fallback: not enough same-view data
        pred_cal = item[key]
        p_cat = np.concatenate([x[key]       for x in train])
        t_cat = np.concatenate([x['truth_3d'] for x in train])
        valid = np.isfinite(p_cat) & np.isfinite(t_cat)
        if valid.sum() > 20:
            try:
                params = fit_curve(p_cat[valid], t_cat[valid], min_count=15)
                pred_cal = apply_calibration(item[key], params)
            except ValueError as e:
                print(f'  calib skip ({label}):', e)
        row = bc.make_row(label, pred_cal, item['truth_3d'], dict(me))
        if row:
            rows_D.append(row)
            append_row(path_D, row)

print(f'\nDone. Rows: A={len(rows_A)}, B={len(rows_B)}, D={len(rows_D)}')

All imports OK
    static right: ROM 30.3deg < 60 - no expected flexion
    static left: ROM 2.5deg < 60 - no expected flexion
[1/70] PM_000_ex1_unknown  0.1s  MAE R/L: nan°/nan°
    static right: ROM 30.3deg < 60 - no expected flexion
    static left: ROM 2.5deg < 60 - no expected flexion
[2/70] PM_000_ex1_cam18  0.1s  MAE R/L: nan°/nan°
    static right: ROM 11.7deg < 60 - no expected flexion
    static left: ROM 3.7deg < 60 - no expected flexion
[3/70] PM_001_ex1_unknown  0.3s  MAE R/L: nan°/nan°
    static right: ROM 11.7deg < 60 - no expected flexion
    static left: ROM 3.7deg < 60 - no expected flexion
[4/70] PM_001_ex1_cam18  0.2s  MAE R/L: nan°/nan°
    static right: ROM 13.5deg < 60 - no expected flexion
    static left: ROM 2.1deg < 60 - no expected flexion
[5/70] PM_002_ex1_unknown  0.1s  MAE R/L: nan°/nan°
    static right: ROM 13.5deg < 60 - no expected flexion
    static left: ROM 2.1deg < 60 - no expected flexion
[6/70] PM_002_ex1_cam18  0.1s  MAE R/L: nan°/nan°
    occ

In [9]:
# ============================================================
# DIAGNOSTIC — why are recordings being skipped?
# ============================================================
import numpy as np

for rec in recordings[:6]:
    lm, world, size, fps = get_3d_landmarks(rec)
    truth_arr = bc.load_truth_array(rec['truth_path'], JOINT_NAMES)
    npz_f0 = int(np.load(os.path.join(
        CACHE, f"{rec['rec_id']}_{rec['f0']}_{rec['f1']}_3d.npz"))['f0'])

    print('='*62)
    print(rec['rec_id'], '|', rec['video_key'])
    print('  truth file   :', os.path.basename(rec['truth_path']))
    print(f'  len truth={len(truth_arr)}  len video={len(world)}  '
          f'ratio={len(truth_arr)/max(len(world),1):.2f}   <-- want ~1.0')

    # --- are missing mocap joints encoded as ZEROS instead of NaN? ---
    idx = [JOINT_NAMES.index(n) for n in ARMS_OVERRIDE['right']]
    sub = truth_arr[:, idx, :3]
    zero_rows = np.all(np.abs(sub) < 1e-6, axis=2).any(axis=1)
    print(f'  truth all-zero joints : {100*zero_rows.mean():5.1f}%   <-- want ~0%')
    print(f'  truth NaN joints      : {100*np.isnan(sub).any(axis=(1,2)).mean():5.1f}%')

    for side in ('right', 'left'):
        arm = ARMS_OVERRIDE[side]
        gt = truth_angle_for(rec, truth_arr, world, npz_f0, fps, arm)

        for thr in (0.70, 0.80):
            lmm = mask_low_visibility(lm, threshold=thr)
            pa = angles_2d.elbow_angle(lmm, side=side, size=size)
            cov = 100*np.isfinite(pa).mean()
            print(f'  {side:5} thr={thr}  pred coverage {cov:5.1f}%', end='')
            if thr == 0.70:
                print(f'   pred range {np.nanmax(pa)-np.nanmin(pa):5.1f}°'
                      if np.isfinite(pa).any() else '   pred ALL NaN')
            else:
                print()

        gtr = (np.nanpercentile(gt,95)-np.nanpercentile(gt,5)
               if np.isfinite(gt).any() else np.nan)
        print(f'  {side:5} gt coverage {100*np.isfinite(gt).mean():5.1f}%  '
              f'gt range {gtr:5.1f}°   <-- <15° = quasi-static')

        # wider lag search, on filtered signals
        lmm = mask_low_visibility(lm, threshold=0.70)
        pa  = angles_2d.elbow_angle(lmm, side=side, size=size)
        if np.isfinite(pa).any() and np.isfinite(gt).any():
            for ml in (30, 150):
                l, r = bc.best_lag(pa, gt, max_lag=ml, filter_signals=True)
                print(f'  {side:5} MAX_LAG={ml:3}  ->  lag={l:4}  r={r:5.2f}')

        # left/right swap test: is MediaPipe's side flipped vs mocap?
        gt_opp = truth_angle_for(rec, truth_arr, world, npz_f0, fps,
                                 ARMS_OVERRIDE['left' if side=='right' else 'right'])
        if np.isfinite(pa).any() and np.isfinite(gt_opp).any():
            _, r_opp = bc.best_lag(pa, gt_opp, max_lag=150, filter_signals=True)
            print(f'  {side:5} same-side r vs OPPOSITE-side r : '
                  f'{bc.best_lag(pa, gt, max_lag=150, filter_signals=True)[1]:.2f}'
                  f'  vs  {r_opp:.2f}   <-- opposite higher = sides swapped')

PM_000_ex1_unknown | PM_000-Camera17-30fps
  truth file   : PM_000-30fps.npy
  len truth=1304  len video=1169  ratio=1.12   <-- want ~1.0
  truth all-zero joints :   0.0%   <-- want ~0%
  truth NaN joints      :   0.0%
  right thr=0.7  pred coverage 100.0%   pred range  35.4°
  right thr=0.8  pred coverage 100.0%
  right gt coverage 100.0%  gt range  30.3°   <-- <15° = quasi-static
  right MAX_LAG= 30  ->  lag=   1  r= 0.77
  right MAX_LAG=150  ->  lag=   1  r= 0.77
  right same-side r vs OPPOSITE-side r : 0.77  vs  0.65   <-- opposite higher = sides swapped
  left  thr=0.7  pred coverage 100.0%   pred range  11.2°
  left  thr=0.8  pred coverage 100.0%
  left  gt coverage 100.0%  gt range   2.5°   <-- <15° = quasi-static
  left  MAX_LAG= 30  ->  lag= -30  r=-0.25
  left  MAX_LAG=150  ->  lag= 129  r= 0.64
  left  same-side r vs OPPOSITE-side r : 0.64  vs  0.52   <-- opposite higher = sides swapped
PM_000_ex1_cam18 | PM_000-Camera18-30fps-transposed
  truth file   : PM_000-30fps.npy
  l

In [38]:
A = df.copy()
A['subj'] = A['view'].str.replace('/cam18', '', regex=False)
print(df.groupby('variant')[['mae','rmse','bias','r','share_gt20']].median().round(2))
print()
print(A[A.variant.isin(['A_2D_Base','D_2D_Calibrated','D_2D_PerView'])]
        .groupby(['variant','cam','subj'])[['mae','bias','r']]
        .agg(['count','median']).round(2))

                   mae   rmse   bias     r  share_gt20
variant                                               
A_2D_Base        16.12  23.83   7.99  0.91       30.40
B_3D_World       21.93  26.56  14.90  0.89       48.69
D_2D_Calibrated  14.19  18.51   3.79  0.93       20.80
D_3D_Calibrated  15.02  19.37   4.46  0.91       25.85

                                     mae         bias            r       
                                   count median count median count median
variant         cam   subj                                               
A_2D_Base       cam17 Front           18  12.00    18  -1.31    18   0.95
                      Half-profile     5  17.70     5   2.24     5   0.92
                      Profile         10  16.77    10  13.18    10   0.85
                cam18 Front            6  24.70     6   8.49     6   0.85
                      Half-profile     4  13.76     4  -3.81     4   0.93
                      Profile         13  20.23    13  19.09    13   0.88
D_2

In [39]:
# ============================================================
# ROM SCAN — which recordings actually flex the elbow?
# (paper p.10: "only ... exercises where there is an expected flexion")
# ============================================================
import numpy as np, pandas as pd

rows = []
for rec in recordings:
    truth_arr = bc.load_truth_array(rec['truth_path'], JOINT_NAMES)
    for side in ('right', 'left'):
        gt = bc.truth_elbow_3d(truth_arr, JOINT_NAMES, ARMS_OVERRIDE[side], use_dims=3)
        if np.isfinite(gt).any():
            rom = float(np.nanpercentile(gt, 95) - np.nanpercentile(gt, 5))
            rows.append((rec['exercise'], side, rec['rec_id'], rom))

df_rom = pd.DataFrame(rows, columns=['ex', 'side', 'rec_id', 'rom'])
print(df_rom.groupby(['ex', 'side'])['rom'].agg(['count', 'median', 'min', 'max']).round(1))
print()
print('usable fraction (ROM >= 15 deg):')
print(df_rom.assign(ok=df_rom.rom >= 15).groupby(['ex', 'side'])['ok'].mean().round(2))
print()
print('TOTAL usable:', int((df_rom.rom >= 15).sum()), '/', len(df_rom))

          count  median   min    max
ex side                             
1  left      26     6.1   2.3   18.8
   right     26    24.3   7.3   50.8
2  left      24   113.5  93.3  133.7
   right     24   115.2  90.7  126.8
3  left      20   105.2  80.4  134.1
   right     20   102.3  81.1  130.1

usable fraction (ROM >= 15 deg):
ex  side 
1   left     0.15
    right    0.62
2   left     1.00
    right    1.00
3   left     1.00
    right    1.00
Name: ok, dtype: float64

TOTAL usable: 108 / 140


In [28]:
# ============================================================
# RESULTS — A vs B (works on old or new CSVs)
# ============================================================
import pandas as pd, glob, os

df = pd.concat([pd.read_csv(f) for f in glob.glob(os.path.join(RESULTS, 'variant_*.csv'))],
               ignore_index=True)

WANT = ['mae', 'rmse', 'bias', 'r', 'coverage', 'share_gt20', 'lag', 'lag_r', 'gt_range']
HAVE = [c for c in WANT if c in df.columns]
MISS = [c for c in WANT if c not in df.columns]

print('rows:', len(df), '| variants:', sorted(df.variant.unique()))
print('columns present :', HAVE)
if MISS:
    print('columns missing :', MISS, ' <-- these CSVs are from the OLD run')
print()

print('--- overall, by variant ---')
print(df.groupby('variant')[HAVE].median().round(2))
print()

if 'ex' in df.columns and df['ex'].notna().any():
    print('--- by variant x exercise ---')
    print(df.groupby(['variant', 'ex'])[[c for c in ['mae','bias','r','coverage'] if c in HAVE]]
            .agg(['count', 'median']).round(2))
    print()
else:
    print("(no usable 'ex' column -- old CSVs wrote it blank; "
          "re-run cell[9] after the meta fix)\n")

if 'view' in df.columns and df['view'].notna().any():
    print('--- by variant x view ---')
    print(df.groupby(['variant', 'view'])[[c for c in ['mae','bias','r'] if c in HAVE]]
            .median().round(2))
    print()

print('--- paper reference ---')
print('  IMU gold standard      RMSE 4.95 deg')
print('  AlphaPose 2D vs 3D GT  MAE  6.75 deg')
print('  MediaPipe 2D vs 3D GT  MAE  9.36 deg   <-- your target')

rows: 224 | variants: ['A_2D_Base', 'B_3D_World', 'D_2D_Calibrated', 'D_3D_Calibrated']
columns present : ['mae', 'rmse', 'bias', 'r', 'coverage', 'share_gt20', 'lag', 'lag_r']
columns missing : ['gt_range']  <-- these CSVs are from the OLD run

--- overall, by variant ---
                   mae   rmse   bias     r  coverage  share_gt20  lag  lag_r
variant                                                                     
A_2D_Base        16.12  23.83   7.99  0.91     98.63       30.40  0.0   0.91
B_3D_World       21.93  26.56  14.90  0.89     98.37       48.69  0.0   0.91
D_2D_Calibrated  14.19  18.51   3.79  0.93     98.63       20.80  0.0   0.91
D_3D_Calibrated  15.02  19.37   4.46  0.91     98.37       25.85  0.0   0.91

--- by variant x exercise ---
                     mae         bias            r        coverage       
                   count median count median count median    count median
variant         ex                                                       
A_2D_Base  

In [23]:
print(pd.crosstab(df[df.variant=='A_2D_Base'].ex,
                  df[df.variant=='A_2D_Base'].view))
print()
print(df[df.variant=='A_2D_Base'].groupby(['ex','cam','view'])['mae'].agg(['count','median']).round(2))

view  Front  Front/cam18  Half-profile  Half-profile/cam18  Profile  \
ex                                                                    
2        14            4             3                   3        0   
3         0            0             0                   0        6   

view  Profile/cam18  
ex                   
2                 0  
3                 9  

                             count  median
ex cam   view                             
2  cam17 Front                  14   11.19
         Half-profile            3   22.35
   cam18 Front/cam18             4   24.11
         Half-profile/cam18      3   12.58
3  cam17 Profile                 6   15.33
   cam18 Profile/cam18           9   20.67


In [29]:
# which recordings fail alignment, and why?
fails = []
for rec in recordings:
    lm, world, size, fps = get_3d_landmarks(rec)
    truth_arr = bc.load_truth_array(rec['truth_path'], JOINT_NAMES)
    npz_f0 = int(np.load(os.path.join(
        CACHE, f"{rec['rec_id']}_{rec['f0']}_{rec['f1']}_3d.npz"))['f0'])
    lmm = mask_low_visibility(lm, threshold=THRESHOLD)
    for side in ('right','left'):
        arm = ARMS_OVERRIDE[side]
        gt = truth_angle_for(rec, truth_arr, world, npz_f0, fps, arm)
        if not np.isfinite(gt).any(): continue
        rom = float(np.nanpercentile(gt,95)-np.nanpercentile(gt,5))
        if rom < MIN_ROM: continue
        pa = angles_2d.elbow_angle(lmm, side=side, size=size)
        cov = 100*np.isfinite(pa).mean()
        lag, r = bc.best_lag(pa, gt, max_lag=MAX_LAG, filter_signals=True)
        fails.append(dict(rec=rec['rec_id'], side=side, ex=rec['exercise'],
                          cam=rec.get('cam_label'), rom=round(rom,1),
                          cov=round(cov,1), lag=lag, r=round(float(r),2),
                          passed=(abs(lag)<MAX_LAG and r>=0.7)))
f = pd.DataFrame(fails)
print(f.groupby(['ex','cam'])['passed'].agg(['count','mean']).round(2))
print()
print(f[~f.passed].sort_values('r').head(15).to_string(index=False))

          count  mean
ex cam               
2  cam17     24  0.96
   cam18     24  0.42
3  cam17     20  0.50
   cam18     20  0.65

               rec  side  ex   cam   rom   cov  lag     r  passed
  PM_003_ex2_cam18 right   2 cam18 104.9   0.0    0 -2.00   False
  PM_013_ex2_cam18 right   2 cam18  87.5   0.0    0 -2.00   False
PM_030_ex3_unknown  left   3 cam17 104.5   0.0    0 -2.00   False
PM_045_ex3_unknown  left   3 cam17 106.5   0.0    0 -2.00   False
PM_108_ex3_unknown  left   3 cam17 105.2   0.0    0 -2.00   False
PM_121_ex3_unknown  left   3 cam17  88.6   0.0    0 -2.00   False
PM_011_ex3_unknown  left   3 cam17 116.9   5.4   13 -0.26   False
PM_107_ex3_unknown  left   3 cam17 102.2   2.1   30 -0.11   False
PM_010_ex3_unknown  left   3 cam17 109.9  11.8  -20  0.02   False
  PM_025_ex2_cam18 right   2 cam18 111.5   0.5    0  0.09   False
PM_119_ex3_unknown  left   3 cam17  90.4  11.9   -1  0.16   False
PM_044_ex3_unknown  left   3 cam17  80.9   6.0  -25  0.22   False
PM_031_ex

In [32]:
from calibration import leave_one_person_out

records = [dict(person=x['meta']['person'], raw=x['pred_2d'], truth=x['truth_3d'])
           for x in aligned_cache]
lopo = leave_one_person_out(records)
print(lopo[['person','n','mae_raw','mae_cal','gt20_raw','gt20_cal',
            'range_raw','range_cal','range_truth']].round(2))
print()
print('range_cal / range_truth =',
      round(lopo.range_cal.median()/lopo.range_truth.median(), 3), '  <-- want ~1.0')
print('mae  :', round(lopo.mae_raw.median(),2), '->', round(lopo.mae_cal.median(),2))
print('gt20 :', round(lopo.gt20_raw.median(),1), '% ->', round(lopo.gt20_cal.median(),1), '%')

    person      n  mae_raw  mae_cal  gt20_raw  gt20_cal  range_raw  range_cal  \
0   PM_003  12827    13.84    15.08     18.09     30.62     126.29     103.57   
1   PM_004   5363    19.52    13.86     35.15     21.14     159.55     108.45   
2   PM_010   6489    12.68    12.49     19.76     19.12     101.71      87.90   
3   PM_011   3779    20.28    19.70     35.06     35.62      78.22      78.99   
4   PM_013   1760     7.67     6.08      0.00      0.00     107.65      92.81   
5   PM_014   9401    17.47    12.17     34.56     15.45     128.77     104.73   
6   PM_025   4747     8.66     6.85     10.43      1.85     141.12     117.53   
7   PM_026   6013    12.59     9.49     16.55      8.81     131.55     103.60   
8   PM_030   3778    19.96    16.31     38.70     29.57      85.31      83.58   
9   PM_031   2465    25.12    20.47     52.21     44.91     104.87      90.88   
10  PM_033  10709    13.22    11.35     19.14     11.10     149.55     102.61   
11  PM_040  10027    20.45  

In [ ]:
import pandas as pd, os
df = pd.read_csv(os.path.join(RESULTS, 'variant_B_3d_world.csv'))
print("Columns:", list(df.columns))
print(df.head(2))

In [ ]:
# ============================================================
# FIX exercise column from recording name + replot
# ============================================================
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os, re

path_A = os.path.join(RESULTS, 'variant_A_2d.csv')
path_B = os.path.join(RESULTS, 'variant_B_3d_world.csv')

dfs = []
for path in (path_A, path_B):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        df = pd.read_csv(path)
        if len(df) > 0:
            dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)

# infer exercise from recording column (e.g. PM_000_ex1_cam17_right)
def get_ex(row):
    rec = str(row.get('recording', ''))
    m = re.search(r'_ex(\d+)_', rec)
    if m: return int(m.group(1))
    # fallback: check video path if available
    vid = str(row.get('video', ''))
    if 'Ex1' in vid or 'ex1' in rec: return 1
    if 'Ex2' in vid or 'ex2' in rec: return 2
    if 'Ex3' in vid or 'ex3' in rec: return 3
    return None

df_all['exercise'] = df_all.apply(get_ex, axis=1)

print("Exercise value counts:")
print(df_all['exercise'].value_counts().sort_index())

ex_names = {1:'Ex1\nElbow Flexion', 2:'Ex2\nArm VW', 3:'Ex3\nTable Push'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# view plot
view_pivot = df_all.groupby(['view','variant'])['mae'].median().unstack()
view_pivot.plot(kind='bar', ax=axes[0], rot=15,
                color=['#E74C3C','#27AE60'])
axes[0].axhline(15, color='orange', linestyle='--', linewidth=1.5,
                label='15° clinical threshold')
axes[0].set_title('Median MAE by Camera View', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Median MAE (degrees)')
axes[0].set_xlabel('')
axes[0].legend(title='Variant')
axes[0].grid(axis='y', alpha=0.3)
for bar in axes[0].patches:
    h = bar.get_height()
    if h > 0:
        axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.3,
                     f'{h:.1f}°', ha='center', va='bottom', fontsize=8)

# exercise plot — now fixed
ex_df = df_all.dropna(subset=['exercise'])
if not ex_df.empty:
    ex_pivot = ex_df.groupby(['exercise','variant'])['mae'].median().unstack()
    ex_pivot.plot(kind='bar', ax=axes[1], rot=0,
                  color=['#E74C3C','#27AE60'])
    axes[1].set_title('Median MAE by Exercise', fontweight='bold', fontsize=12)
    axes[1].set_ylabel('Median MAE (degrees)')
    axes[1].set_xticklabels(
        [ex_names.get(int(float(t.get_text())), t.get_text())
         for t in axes[1].get_xticklabels()])
    axes[1].legend(title='Variant')
    axes[1].grid(axis='y', alpha=0.3)
    axes[1].axhline(15, color='orange', linestyle='--', linewidth=1.5,
                    label='15° threshold')
    for bar in axes[1].patches:
        h = bar.get_height()
        if h > 0:
            axes[1].text(bar.get_x() + bar.get_width()/2, h + 0.3,
                         f'{h:.1f}°', ha='center', va='bottom', fontsize=8)
else:
    axes[1].text(0.5, 0.5, 'Could not infer exercise from recording names',
                 ha='center', va='center', transform=axes[1].transAxes)

plt.suptitle('REHAB24-6: MediaPipe vs OptiTrack 3D Ground Truth\n'
             f'Variant A = 2D pixel  |  Variant B = 3D world landmarks  '
             f'({len(df_all[df_all["variant"]=="B_3D_World"])} recordings)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS,'metrics_with_exercise.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: metrics_with_exercise.png")

# summary
a_df = df_all[df_all['variant']=='A_2D_Base']
b_df = df_all[df_all['variant']=='B_3D_World']
print("\n" + "="*50)
print("MEETING SUMMARY")
print("="*50)
print(f"Recordings        : {len(b_df)} / 70")
print(f"Variant A MAE     : {a_df['mae'].median():.1f}°  median")
print(f"Variant B MAE     : {b_df['mae'].median():.1f}°  median")
imp = a_df['mae'].median() - b_df['mae'].median()
print(f"Improvement       : {imp:.1f}° ({100*imp/a_df['mae'].median():.0f}%)")
if not ex_df.empty:
    print("\nBy exercise:")
    for ex in sorted(ex_df['exercise'].dropna().unique()):
        a_ex = ex_df[(ex_df['variant']=='A_2D_Base')  & (ex_df['exercise']==ex)]['mae'].median()
        b_ex = ex_df[(ex_df['variant']=='B_3D_World') & (ex_df['exercise']==ex)]['mae'].median()
        print(f"  Ex{int(ex)} {ex_names.get(int(ex),'').replace(chr(10),' ')}: "
              f"A={a_ex:.1f}°  B={b_ex:.1f}°  improvement={a_ex-b_ex:.1f}°")

In [ ]:
# infer exercise from the CACHE filenames matched to video
import glob, os, re, pandas as pd, numpy as np

path_A = os.path.join(RESULTS, 'variant_A_2d.csv')
path_B = os.path.join(RESULTS, 'variant_B_3d_world.csv')

dfs = []
for path in (path_A, path_B):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        df = pd.read_csv(path)
        if len(df) > 0:
            dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)

# build a lookup: video_key -> exercise number from cache filenames
# cache files are named like PM_000_ex1_cam17_135_1310_3d.npz
cache_files = glob.glob(os.path.join(CACHE, '*.npz'))
video_to_ex = {}
for p in cache_files:
    name = os.path.basename(p)
    m = re.search(r'(PM_\d+\w*)_ex(\d+)_(cam\d+|unknown)', name)
    if m:
        person  = m.group(1)   # PM_000
        ex_num  = int(m.group(2))
        cam     = m.group(3)
        # match to video column: PM_000-Camera17 or PM_000-Camera18
        video_to_ex[f"{person}_{cam}"] = ex_num

print(f"Built exercise lookup for {len(video_to_ex)} recordings")
print("Sample:", list(video_to_ex.items())[:5])

def get_ex_from_video(row):
    video = str(row.get('video', ''))
    person = video.split('-')[0]   # PM_000
    cam_raw = str(row.get('cam', ''))
    # normalise cam
    if 'cam18' in cam_raw or 'cam18' in video.lower():
        cam = 'cam18'
    elif 'cam17' in cam_raw or 'cam17' in video.lower():
        cam = 'cam17'
    else:
        cam = 'unknown'
    key = f"{person}_{cam}"
    return video_to_ex.get(key, None)

df_all['exercise'] = df_all.apply(get_ex_from_video, axis=1)
print("\nExercise counts:")
print(df_all['exercise'].value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt

ex_names = {1:'Ex1\nElbow Flexion', 2:'Ex2\nArm VW', 3:'Ex3\nTable Push'}
ex_df = df_all.dropna(subset=['exercise'])

if not ex_df.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    ex_pivot = ex_df.groupby(['exercise','variant'])['mae'].median().unstack()
    ex_pivot.plot(kind='bar', ax=ax, rot=0, color=['#E74C3C','#27AE60'])
    ax.set_title('Median MAE by Exercise\nVariant A (2D) vs Variant B (3D World)',
                 fontweight='bold', fontsize=12)
    ax.set_ylabel('Median MAE (degrees)')
    ax.set_xticklabels(
        [ex_names.get(int(float(t.get_text())), t.get_text())
         for t in ax.get_xticklabels()])
    ax.axhline(15, color='orange', linestyle='--', linewidth=1.5,
               label='15° threshold')
    ax.legend(title='Variant')
    ax.grid(axis='y', alpha=0.3)
    for bar in ax.patches:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
                    f'{h:.1f}°', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS, 'mae_by_exercise.png'), dpi=150)
    plt.show()
    print("Saved: mae_by_exercise.png")

    print("\nBy exercise:")
    for ex in sorted(ex_df['exercise'].dropna().unique()):
        a_ex = ex_df[(ex_df['variant']=='A_2D_Base')  &(ex_df['exercise']==ex)]['mae'].median()
        b_ex = ex_df[(ex_df['variant']=='B_3D_World') &(ex_df['exercise']==ex)]['mae'].median()
        print(f"  Ex{int(ex)}: A={a_ex:.1f}°  B={b_ex:.1f}°  improvement={a_ex-b_ex:.1f}°")
else:
    print("Still could not find exercise info — cache files may not match video names")
    print("Cache sample:", [os.path.basename(p) for p in cache_files[:3]])
    print("Video sample:", df_all['video'].head(3).tolist())

In [ ]:
# ============================================================
# FIXED COMPARISON — front view only (cam17), all 3 exercises
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec, os

import angles as angles_2d
from angles_world import elbow_angle_3d, mask_world_like
import benchmark_common as bc
from benchmark_common import elbow_angle_3d_ordered
from benchmark_common import butterworth_filter
from visibility import mask_low_visibility

def fill_1d(arr, max_gap=15):
    arr = np.array(arr, dtype=float)
    if np.isnan(arr).all() or not np.isnan(arr).any():
        return arr
    idx  = np.arange(len(arr))
    good = ~np.isnan(arr)
    filled = arr.copy()
    in_gap, gap_start = False, 0
    for i in range(len(arr)):
        if np.isnan(arr[i]) and not in_gap:
            in_gap, gap_start = True, i
        elif not np.isnan(arr[i]) and in_gap:
            in_gap = False
            if i - gap_start <= max_gap:
                filled[gap_start:i] = np.interp(
                    idx[gap_start:i], idx[good], arr[good])
    return filled

ex_labels = {
    1: 'Elbow Flexion (Ex1)',
    2: 'Arm VW (Ex2)',
    3: 'Table Push-ups (Ex3)'
}

# ── find best cached cam17 recording per exercise ────────────
sample_recs = []
for ex in [1, 2, 3]:
    candidates = [r for r in recordings
                  if r['exercise'] == ex and r['cam'] == 'cam17']
    for rec in candidates:
        cp = os.path.join(CACHE,
             f"{rec['rec_id']}_{rec['f0']}_{rec['f1']}_3d.npz")
        if os.path.exists(cp):
            # verify it has valid arm landmarks
            d = np.load(cp)
            valid = np.isfinite(d['world'][:, 12, 0]).sum()
            if valid > 50:
                sample_recs.append((rec, cp, ex))
                print(f"Ex{ex}: using {rec['rec_id']}  "
                      f"valid frames={valid}/{len(d['world'])}")
                break

print(f"\nTotal recordings to plot: {len(sample_recs)}")

# ── compute everything ───────────────────────────────────────
results = []
for rec, cp, ex in sample_recs:
    d      = np.load(cp)
    lm     = d['lm'];  world = d['world']
    size   = tuple(d['size']);  fps = float(d['fps'])
    npz_f0 = int(d['f0'])

    truth_arr = bc.load_truth_array(rec['truth_path'], JOINT_NAMES)
    gt = truth_angle_for(rec, truth_arr, world, npz_f0, fps, ARMS_OVERRIDE['right'])

    lm_m  = mask_low_visibility(lm,    threshold=THRESHOLD)
    wld_m = mask_world_like(world, lm_m)

    pa_raw = angles_2d.elbow_angle(lm_m, side='right', size=size)
    pa     = fill_1d(pa_raw)
    if np.isfinite(pa).sum() > 15:
        pa = butterworth_filter(pa, fs=fps, cutoff=6.0)

    pb_raw = elbow_angle_3d(wld_m, side='right')
    pb     = fill_1d(pb_raw)
    if np.isfinite(pb).sum() > 15:
        pb = butterworth_filter(pb, fs=fps, cutoff=6.0)

    n = min(len(pa), len(pb), len(gt))
    t = np.arange(n) / fps

    lag_r = bc.best_lag(pb[:n], gt[:n], max_lag=MAX_LAG)
    lag   = lag_r[0] if isinstance(lag_r, tuple) else lag_r
    p_al, t_al = bc.overlap(pb[:n], gt[:n], lag)

    mask_a = np.isfinite(pa[:n]) & np.isfinite(gt[:n])
    mask_b = np.isfinite(p_al)   & np.isfinite(t_al)

    mae_a  = np.mean(np.abs(pa[:n][mask_a]-gt[:n][mask_a])) if mask_a.sum() else np.nan
    mae_b  = np.mean(np.abs(p_al[mask_b]-t_al[mask_b]))     if mask_b.sum() else np.nan
    rmse_b = np.sqrt(np.mean((p_al[mask_b]-t_al[mask_b])**2)) if mask_b.sum() else np.nan
    bias_b = np.mean(p_al[mask_b]-t_al[mask_b])              if mask_b.sum() else np.nan
    r_b    = np.corrcoef(p_al[mask_b],t_al[mask_b])[0,1]     if mask_b.sum()>2 else np.nan
    gt20_b = 100*(np.abs(p_al[mask_b]-t_al[mask_b])>20).mean() if mask_b.sum() else np.nan
    valid_pct = 100*np.isfinite(pb[:n]).sum()/n

    print(f"Ex{ex} {rec['rec_id']}: "
          f"MAE_A={mae_a:.1f}° MAE_B={mae_b:.1f}° "
          f"r={r_b:.3f} valid={valid_pct:.0f}%")

    results.append(dict(
        rec=rec, ex=ex, t=t, n=n, fps=fps,
        gt=gt, pa=pa, pb=pb, p_al=p_al, t_al=t_al,
        mae_a=mae_a, mae_b=mae_b, rmse_b=rmse_b,
        bias_b=bias_b, r_b=r_b, gt20_b=gt20_b,
        valid_pct=valid_pct
    ))

if not results:
    print("\nNo valid results found!")
    print("Check: are there cam17 recordings in the cache?")
    cam17 = [r for r in recordings if r['cam']=='cam17']
    print(f"cam17 recordings in list: {len(cam17)}")
    cached = [r for r in cam17
              if os.path.exists(os.path.join(
                  CACHE, f"{r['rec_id']}_{r['f0']}_{r['f1']}_3d.npz"))]
    print(f"cam17 with cache: {len(cached)}")
else:
    # ── PLOT 1: Time series ────────────────────────────────
    fig, axes = plt.subplots(len(results), 1,
                              figsize=(13, 5*len(results)))
    if len(results)==1: axes=[axes]

    for ax, res in zip(axes, results):
        t  = res['t'];  n = res['n']
        gt = res['gt']; pa= res['pa']; pb= res['pb']

        ax.fill_between(t[:n], gt[:n]-10, gt[:n]+10,
                        alpha=0.10, color='black',
                        label='±10° tolerance')
        ax.plot(t[:len(gt[:n])], gt[:n],
                color='black', lw=2.5, zorder=4,
                label='OptiTrack — Motion Sensor (Ground Truth)')
        ax.plot(t[:n], pa[:n],
                color='#E74C3C', lw=1.5, ls='--', alpha=0.85,
                label=f'Variant A — 2D pixel  MAE={res["mae_a"]:.1f}°')
        ax.plot(t[:n], pb[:n],
                color='#27AE60', lw=1.8, alpha=0.9,
                label=f'Variant B — 3D world  MAE={res["mae_b"]:.1f}°  '
                      f'r={res["r_b"]:.3f}')

        ax2 = ax.twinx()
        err_b = np.abs(pb[:n] - gt[:n])
        ax2.plot(t[:n], err_b, color='#F39C12', lw=0.8, alpha=0.5)
        ax2.set_ylabel('|Error B| (deg)', color='#F39C12', fontsize=8)
        ax2.tick_params(axis='y', labelcolor='#F39C12')
        ax2.set_ylim(0, 60)

        nan_b = np.isnan(pb[:n])
        if nan_b.any():
            ax.fill_between(t[:n], 0, 185, where=nan_b,
                            alpha=0.12, color='grey',
                            label='Lost tracking')

        ax.set_title(
            f"{res['rec']['rec_id']}  —  "
            f"{ex_labels[res['ex']]}  —  Front view (cam17)",
            fontsize=11, fontweight='bold')
        ax.set_xlabel('Time (seconds)')
        ax.set_ylabel('Elbow angle (degrees)')
        ax.set_ylim(0, 190)
        ax.legend(loc='lower right', fontsize=8)
        ax.grid(alpha=0.25)

    plt.suptitle(
        'Predicted vs Motion Sensor (OptiTrack) — Front View\n'
        'Black = Ground Truth  |  Red = 2D (A)  |  '
        'Green = 3D (B)  |  Orange = |Error|',
        fontsize=11, fontweight='bold', y=1.005)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS,'comparison_front_view.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: comparison_front_view.png")

    # ── PLOT 2: Scatter per exercise ───────────────────────
    fig, axes = plt.subplots(1, len(results), figsize=(5*len(results), 5))
    if len(results)==1: axes=[axes]

    for ax, res in zip(axes, results):
        mask = np.isfinite(res['p_al']) & np.isfinite(res['t_al'])
        if mask.sum() < 5:
            ax.set_title(f"{ex_labels[res['ex']]}\nNo data")
            continue
        pp = res['p_al'][mask]
        tt = res['t_al'][mask]
        ax.scatter(tt, pp, alpha=0.15, s=3,
                   color='#27AE60', rasterized=True)
        lims = [max(0, min(tt.min(),pp.min())-5),
                min(185, max(tt.max(),pp.max())+5)]
        ax.plot(lims, lims, 'k--', lw=1.5, label='Perfect')
        ax.plot(lims, [l+15 for l in lims],
                color='orange', lw=1, ls=':', alpha=0.7)
        ax.plot(lims, [l-15 for l in lims],
                color='orange', lw=1, ls=':', alpha=0.7,
                label='±15°')
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_xlabel('OptiTrack (degrees)')
        ax.set_ylabel('Variant B predicted (degrees)')
        ax.set_title(
            f"{ex_labels[res['ex']]}\n"
            f"r={res['r_b']:.3f}  MAE={res['mae_b']:.1f}°",
            fontweight='bold')
        ax.legend(fontsize=8); ax.grid(alpha=0.25)

    plt.suptitle('Scatter: Predicted vs OptiTrack — Front View\n'
                 'Dots on dashed line = perfect agreement',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS,'scatter_front_view.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: scatter_front_view.png")

    # ── Print evaluation matrix ─────────────────────────────
    print("\n" + "="*65)
    print("EVALUATION MATRIX — Variant B vs OptiTrack (Front View)")
    print("="*65)
    print(f"{'Recording':<28} {'Ex':>3} {'MAE_A':>6} "
          f"{'MAE_B':>6} {'RMSE':>6} {'Bias':>6} "
          f"{'r':>6} {'>20%':>6} {'Valid%':>7}")
    print("-"*65)
    for res in results:
        print(f"{res['rec']['rec_id']:<28} "
              f"{res['ex']:>3} {res['mae_a']:>6.1f} "
              f"{res['mae_b']:>6.1f} {res['rmse_b']:>6.1f} "
              f"{res['bias_b']:>6.1f} {res['r_b']:>6.3f} "
              f"{res['gt20_b']:>6.1f} {res['valid_pct']:>7.1f}")
    print("-"*65)

    valid_r = [res['r_b']    for res in results if not np.isnan(res['r_b'])]
    valid_m = [res['mae_b']  for res in results if not np.isnan(res['mae_b'])]
    valid_a = [res['mae_a']  for res in results if not np.isnan(res['mae_a'])]
    if valid_m:
        print(f"\nMean MAE_A : {np.mean(valid_a):.1f}°")
        print(f"Mean MAE_B : {np.mean(valid_m):.1f}°")
        imp = np.mean(valid_a) - np.mean(valid_m)
        print(f"Improvement: {imp:.1f}° ({100*imp/np.mean(valid_a):.0f}%)")
        print(f"Mean r     : {np.mean(valid_r):.3f}")

In [ ]:
# debug — see what cam values exist in recordings
cams = [r['cam'] for r in recordings]
print("Unique cam values:", set(cams))
print("Sample rec_ids:", [r['rec_id'] for r in recordings[:6]])
print("Sample video keys:", [r['video_key'] for r in recordings[:6]])

In [ ]:
# fix cam labels
for rec in recordings:
    vk = rec['video_key'].lower()   # lowercase so Camera17 -> camera17
    if 'camera18' in vk or 'transposed' in vk:
        rec['cam'] = 'cam18'
    elif 'camera17' in vk:
        rec['cam'] = 'cam17'
    else:
        rec['cam'] = 'unknown'

from collections import Counter
print("Cam counts:", Counter(r['cam'] for r in recordings))
print("cam17 sample:", [r['rec_id'] for r in recordings if r['cam']=='cam17'][:5])

In [ ]:
# fix both cam and rec_id to match cache filenames
for rec in recordings:
    vk = rec['video_key'].lower()
    if 'camera18' in vk or 'transposed' in vk:
        new_cam = 'cam18'
    elif 'camera17' in vk:
        new_cam = 'cam17'
    else:
        new_cam = 'unknown'

    # rebuild rec_id using original unknown label to match cache files
    # cache was saved with 'unknown' for cam17, so keep that for lookup
    person_id = rec['video_key'].split('-')[0]   # PM_000
    ex        = rec['exercise']
    # the cache file was named with whatever cam was set when Step 5 ran
    # check both possible names
    old_id = rec['rec_id']
    rec['cam'] = new_cam

# verify by checking which cache files actually exist
import glob, os
cached = {os.path.basename(p) for p in glob.glob(f'{CACHE}/*.npz')}
print(f"Total cache files: {len(cached)}")
print("Sample cache names:", list(cached)[:5])
print()

# now find cam17 recordings whose cache EXISTS under the 'unknown' name
cam17_with_cache = []
for rec in recordings:
    if rec['cam'] != 'cam17':
        continue
    # try both rec_id variants
    for label in ['unknown', 'cam17']:
        person = rec['video_key'].split('-')[0]
        test_id = f"{person}_ex{rec['exercise']}_{label}"
        cache_name = f"{test_id}_{rec['f0']}_{rec['f1']}_3d.npz"
        if cache_name in cached:
            rec['rec_id']    = test_id   # update to matching name
            rec['cache_key'] = cache_name
            cam17_with_cache.append(rec)
            break

print(f"cam17 recordings with matching cache: {len(cam17_with_cache)}")
for r in cam17_with_cache[:5]:
    print(f"  {r['rec_id']}  cache={r['cache_key']}")

In [ ]:
sample_recs = []
for ex in [1, 2, 3]:
    candidates = [r for r in cam17_with_cache if r['exercise'] == ex]
    for rec in candidates:
        cp = os.path.join(CACHE, rec['cache_key'])
        if os.path.exists(cp):
            d = np.load(cp)
            valid = np.isfinite(d['world'][:, 12, 0]).sum()
            if valid > 50:
                sample_recs.append((rec, cp, ex))
                print(f"Ex{ex}: {rec['rec_id']}  valid={valid}/{len(d['world'])}")
                break

print(f"\nRecordings ready to plot: {len(sample_recs)}")

In [24]:
# ============================================================
# FINAL COMPARISON PLOTS — predicted vs motion sensor
# ============================================================
import numpy as np, matplotlib.pyplot as plt, os

import angles as angles_2d
from angles_world import elbow_angle_3d, mask_world_like
import benchmark_common as bc
from benchmark_common import elbow_angle_3d_ordered
from benchmark_common import butterworth_filter
from visibility import mask_low_visibility

def fill_1d(arr, max_gap=15):
    arr = np.array(arr, dtype=float)
    if np.isnan(arr).all() or not np.isnan(arr).any():
        return arr
    idx  = np.arange(len(arr))
    good = ~np.isnan(arr)
    filled = arr.copy()
    in_gap, gap_start = False, 0
    for i in range(len(arr)):
        if np.isnan(arr[i]) and not in_gap:
            in_gap, gap_start = True, i
        elif not np.isnan(arr[i]) and in_gap:
            in_gap = False
            if i - gap_start <= max_gap:
                filled[gap_start:i] = np.interp(
                    idx[gap_start:i], idx[good], arr[good])
    return filled

ex_labels = {1:'Elbow Flexion (Ex1)',
             2:'Arm VW (Ex2)',
             3:'Table Push-ups (Ex3)'}

# ── compute predictions + metrics ───────────────────────────
results = []
for rec, cp, ex in sample_recs:
    d      = np.load(cp)
    lm     = d['lm'];  world = d['world']
    size   = tuple(d['size']);  fps = float(d['fps'])
    npz_f0 = int(d['f0'])

    truth_arr = bc.load_truth_array(rec['truth_path'], JOINT_NAMES)
    gt = truth_angle_for(rec, truth_arr, world, npz_f0, fps, ARMS_OVERRIDE['right'])

    lm_m  = mask_low_visibility(lm, threshold=THRESHOLD)
    wld_m = mask_world_like(world, lm_m)

    pa_raw = angles_2d.elbow_angle(lm_m, side='right', size=size)
    pa     = fill_1d(pa_raw)
    if np.isfinite(pa).sum() > 15:
        pa = butterworth_filter(pa, fs=fps, cutoff=6.0)

    pb_raw = elbow_angle_3d(wld_m, side='right')
    pb     = fill_1d(pb_raw)
    if np.isfinite(pb).sum() > 15:
        pb = butterworth_filter(pb, fs=fps, cutoff=6.0)

    n = min(len(pa), len(pb), len(gt))
    t = np.arange(n) / fps

    lag_r = bc.best_lag(pb[:n], gt[:n], max_lag=MAX_LAG)
    lag   = lag_r[0] if isinstance(lag_r, tuple) else lag_r
    p_al, t_al = bc.overlap(pb[:n], gt[:n], lag)

    mask_a = np.isfinite(pa[:n]) & np.isfinite(gt[:n])
    mask_b = np.isfinite(p_al)   & np.isfinite(t_al)

    mae_a  = np.mean(np.abs(pa[:n][mask_a]-gt[:n][mask_a])) if mask_a.sum() else np.nan
    mae_b  = np.mean(np.abs(p_al[mask_b]-t_al[mask_b]))     if mask_b.sum() else np.nan
    rmse_b = np.sqrt(np.mean((p_al[mask_b]-t_al[mask_b])**2)) if mask_b.sum() else np.nan
    bias_b = np.mean(p_al[mask_b]-t_al[mask_b])               if mask_b.sum() else np.nan
    r_b    = np.corrcoef(p_al[mask_b],t_al[mask_b])[0,1]      if mask_b.sum()>2 else np.nan
    gt20_b = 100*(np.abs(p_al[mask_b]-t_al[mask_b])>20).mean() if mask_b.sum() else np.nan
    valid_pct = 100*np.isfinite(pb[:n]).sum()/n

    results.append(dict(
        rec=rec, ex=ex, t=t, n=n, fps=fps,
        gt=gt, pa=pa, pb=pb, p_al=p_al, t_al=t_al,
        mae_a=mae_a, mae_b=mae_b, rmse_b=rmse_b,
        bias_b=bias_b, r_b=r_b, gt20_b=gt20_b,
        valid_pct=valid_pct
    ))
    print(f"Ex{ex}: MAE_A={mae_a:.1f}°  MAE_B={mae_b:.1f}°  "
          f"r={r_b:.3f}  valid={valid_pct:.0f}%")

# ── PLOT 1: Time series (3 subplots) ─────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 15))

for ax, res in zip(axes, results):
    t  = res['t'];  n = res['n']
    gt = res['gt']; pa= res['pa']; pb= res['pb']

    ax.fill_between(t[:n], gt[:n]-10, gt[:n]+10,
                    alpha=0.10, color='black',
                    label='±10° tolerance band')
    ax.plot(t[:n], gt[:n], color='black', lw=2.5, zorder=4,
            label='OptiTrack — Motion Sensor (Ground Truth)')
    ax.plot(t[:n], pa[:n], color='#E74C3C', lw=1.5,
            ls='--', alpha=0.85,
            label=f'Variant A — 2D pixel  MAE={res["mae_a"]:.1f}°')
    ax.plot(t[:n], pb[:n], color='#27AE60', lw=1.8, alpha=0.9,
            label=f'Variant B — 3D world  MAE={res["mae_b"]:.1f}°  '
                  f'r={res["r_b"]:.3f}')

    ax2 = ax.twinx()
    ax2.plot(t[:n], np.abs(pb[:n]-gt[:n]),
             color='#F39C12', lw=0.8, alpha=0.5)
    ax2.set_ylabel('|Error B| (deg)', color='#F39C12', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='#F39C12')
    ax2.set_ylim(0, 60)

    nan_b = np.isnan(pb[:n])
    if nan_b.any():
        ax.fill_between(t[:n], 0, 185, where=nan_b,
                        alpha=0.12, color='grey',
                        label='Lost tracking')

    ax.set_title(
        f"{ex_labels[res['ex']]}  —  {res['rec']['rec_id']}  —  Front view",
        fontsize=11, fontweight='bold')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Elbow angle (degrees)')
    ax.set_ylim(0, 190)
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.25)

plt.suptitle(
    'Predicted vs Motion Sensor (OptiTrack) Elbow Angle — Front View\n'
    'Black = Ground Truth  |  Red dashed = 2D (A)  |  '
    'Green = 3D (B)  |  Orange = |Error B|',
    fontsize=12, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS,'final_timeseries.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: final_timeseries.png")

# ── PLOT 2: Scatter (3 subplots) ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, res in zip(axes, results):
    mask = np.isfinite(res['p_al']) & np.isfinite(res['t_al'])
    if mask.sum() < 5:
        ax.set_title(f"{ex_labels[res['ex']]}\nNo data")
        continue
    pp = res['p_al'][mask]
    tt = res['t_al'][mask]
    ax.scatter(tt, pp, alpha=0.08, s=3,
               color='#27AE60', rasterized=True)
    lims = [max(0,  min(tt.min(), pp.min()) - 5),
            min(185, max(tt.max(), pp.max()) + 5)]
    ax.plot(lims, lims, 'k--', lw=2, label='Perfect agreement')
    ax.plot(lims, [l+15 for l in lims],
            color='orange', lw=1.2, ls=':', alpha=0.8)
    ax.plot(lims, [l-15 for l in lims],
            color='orange', lw=1.2, ls=':', alpha=0.8,
            label='±15° band')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('OptiTrack angle (degrees)', fontsize=9)
    ax.set_ylabel('Variant B predicted (degrees)', fontsize=9)
    ax.set_title(
        f"{ex_labels[res['ex']]}\n"
        f"r = {res['r_b']:.3f}  MAE = {res['mae_b']:.1f}°",
        fontweight='bold', fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.25)

plt.suptitle(
    'Scatter: Variant B (3D World) Predicted vs OptiTrack Ground Truth\n'
    'Each dot = one frame  |  Black dashed = perfect  |  Orange = ±15°',
    fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS,'final_scatter.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: final_scatter.png")

# ── Print full evaluation matrix ──────────────────────────────
print("\n" + "="*68)
print("FULL EVALUATION MATRIX — Predicted vs OptiTrack (Front View)")
print("="*68)
print(f"{'Exercise':<22} {'MAE_A':>6} {'MAE_B':>6} {'RMSE_B':>7} "
      f"{'Bias_B':>7} {'r_B':>7} {'>20%':>6} {'Valid%':>7}")
print("-"*68)
for res in results:
    print(f"{ex_labels[res['ex']]:<22} "
          f"{res['mae_a']:>6.1f} {res['mae_b']:>6.1f} "
          f"{res['rmse_b']:>7.1f} {res['bias_b']:>7.1f} "
          f"{res['r_b']:>7.3f} {res['gt20_b']:>6.1f} "
          f"{res['valid_pct']:>7.1f}")
print("-"*68)

valid_r  = [r['r_b']    for r in results if not np.isnan(r['r_b'])]
valid_mb = [r['mae_b']  for r in results if not np.isnan(r['mae_b'])]
valid_ma = [r['mae_a']  for r in results if not np.isnan(r['mae_a'])]
print(f"{'MEAN':<22} {np.mean(valid_ma):>6.1f} {np.mean(valid_mb):>6.1f} "
      f"{np.mean([r['rmse_b'] for r in results]):>7.1f} "
      f"{np.mean([r['bias_b'] for r in results]):>7.1f} "
      f"{np.mean(valid_r):>7.3f}")

print("\n" + "="*68)
print("COLUMN DEFINITIONS")
print("="*68)
print("MAE_A   Mean Absolute Error — Variant A 2D pixel (degrees)")
print("MAE_B   Mean Absolute Error — Variant B 3D world (degrees)")
print("RMSE_B  Root Mean Square Error — Variant B (degrees)")
print("Bias_B  Systematic offset: negative = under-reads vs OptiTrack")
print("r_B     Pearson correlation: 1.0 = perfect tracking")
print(">20%    % of frames where |error| exceeds 20 degrees")
print("Valid%  % of frames where MediaPipe detected the arm")

print("\n" + "="*68)
print("KEY NUMBERS FOR PROGRESS MEETING")
print("="*68)
imp = np.mean(valid_ma) - np.mean(valid_mb)
print(f"Variant A (2D) mean MAE  : {np.mean(valid_ma):.1f}°")
print(f"Variant B (3D) mean MAE  : {np.mean(valid_mb):.1f}°")
print(f"Improvement A → B        : {imp:.1f}° ({100*imp/np.mean(valid_ma):.0f}%)")
print(f"Mean Pearson r (B vs GT) : {np.mean(valid_r):.3f}")
under15 = sum(1 for r in results if not np.isnan(r['mae_b']) and r['mae_b']<15)
print(f"Exercises under 15° MAE  : {under15}/{len(results)}")

NameError: name 'sample_recs' is not defined

In [ ]:
# check cam18 cache status
import os, glob, numpy as np

cam18_recs = [r for r in recordings if r['cam'] == 'cam18']
print(f"Total cam18 recordings: {len(cam18_recs)}")

cam18_cached = []
for rec in cam18_recs:
    cp = os.path.join(CACHE, f"{rec['rec_id']}_{rec['f0']}_{rec['f1']}_3d.npz")
    if os.path.exists(cp):
        d     = np.load(cp)
        valid = np.isfinite(d['world'][:, 12, 0]).sum()
        total = len(d['world'])
        cam18_cached.append({
            'rec': rec, 'cp': cp,
            'valid': valid, 'total': total,
            'valid_pct': 100*valid/total
        })

print(f"cam18 with cache: {len(cam18_cached)}")
print()
print(f"{'rec_id':<30} {'ex':>3} {'valid':>7} {'total':>7} {'valid%':>7}")
print("-"*55)
for c in sorted(cam18_cached, key=lambda x: x['rec']['exercise']):
    print(f"{c['rec']['rec_id']:<30} "
          f"{c['rec']['exercise']:>3} "
          f"{c['valid']:>7} "
          f"{c['total']:>7} "
          f"{c['valid_pct']:>7.1f}%")

In [ ]:
# pick best cam18 recording per exercise (highest valid%)
best_cam18 = {}
for c in cam18_cached:
    ex = c['rec']['exercise']
    if ex not in best_cam18 or c['valid_pct'] > best_cam18[ex]['valid_pct']:
        best_cam18[ex] = c

print("Best cam18 recording per exercise:")
for ex, c in sorted(best_cam18.items()):
    print(f"  Ex{ex}: {c['rec']['rec_id']}  "
          f"valid={c['valid_pct']:.1f}%  "
          f"path={c['cp']}")

In [ ]:
# ============================================================
# SIDE VIEW (cam18) FINAL — left arm, safe smoothing
# ============================================================
import numpy as np, matplotlib.pyplot as plt, os

import angles as angles_2d
from angles_world import elbow_angle_3d, mask_world_like
import benchmark_common as bc
from benchmark_common import elbow_angle_3d_ordered
from benchmark_common import butterworth_filter
from visibility import mask_low_visibility

def fill_1d(arr, max_gap=15):
    arr = np.array(arr, dtype=float)
    if np.isnan(arr).all() or not np.isnan(arr).any():
        return arr
    idx  = np.arange(len(arr))
    good = ~np.isnan(arr)
    filled = arr.copy()
    in_gap, gap_start = False, 0
    for i in range(len(arr)):
        if np.isnan(arr[i]) and not in_gap:
            in_gap, gap_start = True, i
        elif not np.isnan(arr[i]) and in_gap:
            in_gap = False
            if i - gap_start <= max_gap:
                filled[gap_start:i] = np.interp(
                    idx[gap_start:i], idx[good], arr[good])
    return filled

def safe_smooth(arr, fps, cutoff=6.0):
    """Butterworth filter with fallback — returns fill_1d output if filter fails."""
    if np.isfinite(arr).sum() < 15:
        return arr
    try:
        smoothed = butterworth_filter(arr, fs=fps, cutoff=cutoff)
        if np.isfinite(smoothed).sum() == 0:
            return arr   # filter failed silently — use unsmoothed
        return smoothed
    except Exception:
        return arr

ex_labels = {1:'Elbow Flexion (Ex1)',
             2:'Arm VW (Ex2)',
             3:'Table Push-ups (Ex3)'}

SIDE_CAM18 = 'left'
ARMS_LEFT  = ('7: LeftArm', '8: LeftForeArm', '9: LeftHand')

sample_cam18 = [
    (c['rec'], c['cp'], c['rec']['exercise'])
    for c in sorted(best_cam18.values(),
                    key=lambda x: x['rec']['exercise'])
]

print(f"Plotting {len(sample_cam18)} side-view recordings (LEFT arm)")

results_side = []
for rec, cp, ex in sample_cam18:
    d      = np.load(cp)
    lm     = d['lm'];  world = d['world']
    size   = tuple(d['size']);  fps = float(d['fps'])
    npz_f0 = int(d['f0'])

    truth_arr = bc.load_truth_array(rec['truth_path'], JOINT_NAMES)
    gt = truth_angle_for(rec, truth_arr, world, npz_f0, fps, ARMS_LEFT)

    lm_m  = mask_low_visibility(lm,    threshold=THRESHOLD)
    wld_m = mask_world_like(world, lm_m)

    pa_raw = angles_2d.elbow_angle(lm_m, side=SIDE_CAM18, size=size)
    pa     = safe_smooth(fill_1d(pa_raw), fps)

    pb_raw = elbow_angle_3d(wld_m, side=SIDE_CAM18)
    pb     = safe_smooth(fill_1d(pb_raw), fps)

    n = min(len(pa), len(pb), len(gt))
    t = np.arange(n) / fps

    valid_pct = 100 * np.isfinite(pb[:n]).sum() / n if n > 0 else 0

    lag_r = bc.best_lag(pb[:n], gt[:n], max_lag=MAX_LAG)
    lag   = lag_r[0] if isinstance(lag_r, tuple) else lag_r
    p_al, t_al = bc.overlap(pb[:n], gt[:n], lag)

    mask_a = np.isfinite(pa[:n]) & np.isfinite(gt[:n])
    mask_b = np.isfinite(p_al)   & np.isfinite(t_al)

    mae_a  = np.mean(np.abs(pa[:n][mask_a]-gt[:n][mask_a])) if mask_a.sum() else np.nan
    mae_b  = np.mean(np.abs(p_al[mask_b]-t_al[mask_b]))     if mask_b.sum() else np.nan
    rmse_b = np.sqrt(np.mean((p_al[mask_b]-t_al[mask_b])**2)) if mask_b.sum() else np.nan
    bias_b = np.mean(p_al[mask_b]-t_al[mask_b])               if mask_b.sum() else np.nan
    r_b    = np.corrcoef(p_al[mask_b],t_al[mask_b])[0,1]      if mask_b.sum()>2 else np.nan
    gt20_b = 100*(np.abs(p_al[mask_b]-t_al[mask_b])>20).mean() if mask_b.sum() else np.nan

    print(f"Ex{ex} {rec['rec_id']}: valid={valid_pct:.0f}%  "
          f"MAE_A={mae_a:.1f}°  MAE_B={mae_b:.1f}°  r={r_b:.3f}")

    results_side.append(dict(
        rec=rec, ex=ex, t=t, n=n, fps=fps,
        gt=gt, pa=pa, pb=pb, p_al=p_al, t_al=t_al,
        mae_a=mae_a, mae_b=mae_b, rmse_b=rmse_b,
        bias_b=bias_b, r_b=r_b, gt20_b=gt20_b,
        valid_pct=valid_pct
    ))

# ── time series plots ─────────────────────────────────────────
n_plots = len(results_side)
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 5*n_plots))
if n_plots == 1: axes = [axes]

for ax, res in zip(axes, results_side):
    t  = res['t'];  n = res['n']
    gt = res['gt']; pa= res['pa']; pb= res['pb']

    ax.fill_between(t[:n], gt[:n]-10, gt[:n]+10,
                    alpha=0.10, color='black', label='±10° tolerance')
    ax.plot(t[:n], gt[:n], color='black', lw=2.5, zorder=4,
            label='OptiTrack — Ground Truth (Left arm)')

    if np.isfinite(pa[:n]).sum() > 10:
        ax.plot(t[:n], pa[:n], color='#E74C3C', lw=1.5,
                ls='--', alpha=0.85,
                label=f'Variant A — 2D  MAE={res["mae_a"]:.1f}°')
    else:
        ax.text(0.05, 0.65,
                'Variant A (2D) — no valid frames',
                transform=ax.transAxes, color='#E74C3C', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='#FADBD8', alpha=0.8))

    if np.isfinite(pb[:n]).sum() > 10:
        ax.plot(t[:n], pb[:n], color='#27AE60', lw=1.8, alpha=0.9,
                label=f'Variant B — 3D  MAE={res["mae_b"]:.1f}°  '
                      f'r={res["r_b"]:.3f}')
    else:
        ax.text(0.05, 0.50,
                'Variant B (3D) — no valid frames',
                transform=ax.transAxes, color='#27AE60', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='#D5F5E3', alpha=0.8))

    nan_b = np.isnan(pb[:n])
    if nan_b.any():
        ax.fill_between(t[:n], 0, 185, where=nan_b,
                        alpha=0.15, color='grey',
                        label=f'Lost tracking ({100*nan_b.sum()/n:.0f}%)')

    ax2 = ax.twinx()
    if np.isfinite(pb[:n]).sum() > 10:
        ax2.plot(t[:n], np.abs(pb[:n]-gt[:n]),
                 color='#F39C12', lw=0.8, alpha=0.5)
    ax2.set_ylabel('|Error B| (deg)', color='#F39C12', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='#F39C12')
    ax2.set_ylim(0, 60)

    ax.set_title(
        f"{ex_labels[res['ex']]}  —  {res['rec']['rec_id']}  "
        f"—  Side view (cam18)  —  Left arm  —  valid={res['valid_pct']:.0f}%",
        fontsize=11, fontweight='bold')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Elbow angle (degrees)')
    ax.set_ylim(0, 190)
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.25)

plt.suptitle(
    'Side View (cam18) — Left Arm: Predicted vs OptiTrack\n'
    'Black = Ground Truth  |  Red dashed = 2D (A)  |  '
    'Green = 3D (B)  |  Orange = |Error|',
    fontsize=12, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS,'sideview_left_arm.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: sideview_left_arm.png")

# ── scatter plots ─────────────────────────────────────────────
fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 5))
if n_plots == 1: axes = [axes]

for ax, res in zip(axes, results_side):
    mask = np.isfinite(res['p_al']) & np.isfinite(res['t_al'])
    if mask.sum() < 5:
        ax.set_title(f"{ex_labels[res['ex']]}\nNo data")
        continue
    pp = res['p_al'][mask]
    tt = res['t_al'][mask]
    ax.scatter(tt, pp, alpha=0.08, s=3,
               color='#27AE60', rasterized=True)
    lims = [max(0,  min(tt.min(), pp.min())-5),
            min(185, max(tt.max(), pp.max())+5)]
    ax.plot(lims, lims, 'k--', lw=2, label='Perfect agreement')
    ax.plot(lims, [l+15 for l in lims], color='orange',
            lw=1.2, ls=':', alpha=0.8)
    ax.plot(lims, [l-15 for l in lims], color='orange',
            lw=1.2, ls=':', alpha=0.8, label='±15°')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('OptiTrack angle (degrees)', fontsize=9)
    ax.set_ylabel('Variant B predicted (degrees)', fontsize=9)
    ax.set_title(
        f"{ex_labels[res['ex']]}\nr={res['r_b']:.3f}  MAE={res['mae_b']:.1f}°",
        fontweight='bold', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)

plt.suptitle(
    'Scatter: Variant B (3D) vs OptiTrack — Side View (Left Arm)\n'
    'Each dot = one frame  |  Black dashed = perfect  |  Orange = ±15°',
    fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS,'sideview_scatter.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: sideview_scatter.png")

# ── front vs side summary table ───────────────────────────────
print("\n" + "="*72)
print("KEY RESULT: FRONT VIEW vs SIDE VIEW")
print("="*72)
print(f"{'Exercise':<22} {'Front_A':>8} {'Front_B':>8} "
      f"{'Side_A':>8} {'Side_B':>8} {'B_improvement':>14}")
print("-"*72)
front_by_ex = {res['ex']: res for res in results}
for res_s in results_side:
    ex  = res_s['ex']
    f   = front_by_ex.get(ex)
    maf = f['mae_a'] if f and not np.isnan(f['mae_a']) else np.nan
    mbf = f['mae_b'] if f and not np.isnan(f['mae_b']) else np.nan
    mas = res_s['mae_a']
    mbs = res_s['mae_b']
    imp = mas - mbs if not np.isnan(mas) and not np.isnan(mbs) else np.nan
    print(f"{ex_labels[ex]:<22} "
          f"{maf:>8.1f}° {mbf:>8.1f}° "
          f"{mas:>8.1f}° {mbs:>8.1f}° "
          f"{imp:>+14.1f}°")

print()
print("B_improvement = Side MAE_A - Side MAE_B")
print("Positive = Variant B (3D) beats Variant A (2D) from side view")

In [ ]:
import numpy as np
from visibility import mask_low_visibility
from angles_world import mask_world_like

# check Ex1 cam18
c = best_cam18[1]
d = np.load(c['cp'])
lm    = d['lm']
world = d['world']

# left arm joints: shoulder=11, elbow=13, wrist=15
print("=== RAW world landmarks (before masking) ===")

print(f"Left shoulder (11) valid: {np.isfinite(world[:,11,0]).sum()}/{len(world)}")
print(f"Left elbow    (13) valid: {np.isfinite(world[:,13,0]).sum()}/{len(world)}")
print(f"Left wrist    (15) valid: {np.isfinite(world[:,15,0]).sum()}/{len(world)}")

print("\n=== VISIBILITY scores (lm[:,joint,3]) ===")
print(f"Left shoulder vis mean: {np.nanmean(lm[:,11,3]):.3f}")
print(f"Left elbow    vis mean: {np.nanmean(lm[:,13,3]):.3f}")
print(f"Left wrist    vis mean: {np.nanmean(lm[:,15,3]):.3f}")
print(f"Threshold used        : {THRESHOLD}")

print("\n=== AFTER masking ===")
lm_m  = mask_low_visibility(lm, threshold=THRESHOLD)
wld_m = mask_world_like(world, lm_m)
print(f"Left shoulder (11) valid after mask: {np.isfinite(wld_m[:,11,0]).sum()}/{len(world)}")
print(f"Left elbow    (13) valid after mask: {np.isfinite(wld_m[:,13,0]).sum()}/{len(world)}")
print(f"Left wrist    (15) valid after mask: {np.isfinite(wld_m[:,15,0]).sum()}/{len(world)}")

from angles_world import elbow_angle_3d
ang = elbow_angle_3d(wld_m, side='left')
print(f"\nLeft arm angle finite frames: {np.isfinite(ang).sum()}/{len(ang)}")
print(f"Sample angles: {ang[:5]}")

In [ ]:
# ============================================================
# SIDE VIEW (cam18) FIXED — use left arm, correct valid%
# ============================================================
import numpy as np, matplotlib.pyplot as plt, os

import angles as angles_2d
from angles_world import elbow_angle_3d, mask_world_like
import benchmark_common as bc
from benchmark_common import elbow_angle_3d_ordered
from benchmark_common import butterworth_filter
from visibility import mask_low_visibility

def fill_1d(arr, max_gap=15):
    arr = np.array(arr, dtype=float)
    if np.isnan(arr).all() or not np.isnan(arr).any():
        return arr
    idx  = np.arange(len(arr))
    good = ~np.isnan(arr)
    filled = arr.copy()
    in_gap, gap_start = False, 0
    for i in range(len(arr)):
        if np.isnan(arr[i]) and not in_gap:
            in_gap, gap_start = True, i
        elif not np.isnan(arr[i]) and in_gap:
            in_gap = False
            if i - gap_start <= max_gap:
                filled[gap_start:i] = np.interp(
                    idx[gap_start:i], idx[good], arr[good])
    return filled

ex_labels = {1:'Elbow Flexion (Ex1)',
             2:'Arm VW (Ex2)',
             3:'Table Push-ups (Ex3)'}

SIDE_CAM18 = 'left'
ARMS_LEFT  = ('7: LeftArm', '8: LeftForeArm', '9: LeftHand')

sample_cam18 = [(c['rec'], c['cp'], c['rec']['exercise'])
                for c in sorted(best_cam18.values(),
                                key=lambda x: x['rec']['exercise'])]

print(f"Plotting {len(sample_cam18)} side-view recordings (LEFT arm)")

results_side = []
for rec, cp, ex in sample_cam18:
    d      = np.load(cp)
    lm     = d['lm'];  world = d['world']
    size   = tuple(d['size']);  fps = float(d['fps'])
    npz_f0 = int(d['f0'])

    truth_arr = bc.load_truth_array(rec['truth_path'], JOINT_NAMES)
    gt = truth_angle_for(rec, truth_arr, world, npz_f0, fps, ARMS_LEFT)

    lm_m  = mask_low_visibility(lm,    threshold=THRESHOLD)
    wld_m = mask_world_like(world, lm_m)

    # Variant A — LEFT arm 2D
    pa_raw = angles_2d.elbow_angle(lm_m, side=SIDE_CAM18, size=size)
    pa     = fill_1d(pa_raw)
    if np.isfinite(pa).sum() > 15:
        pa = butterworth_filter(pa, fs=fps, cutoff=6.0)

    # Variant B — LEFT arm 3D
    pb_raw = elbow_angle_3d(wld_m, side=SIDE_CAM18)
    pb     = fill_1d(pb_raw)
    if np.isfinite(pb).sum() > 15:
        pb = butterworth_filter(pb, fs=fps, cutoff=6.0)

    n = min(len(pa), len(pb), len(gt))
    t = np.arange(n) / fps

    # FIXED: valid_pct from pb not world[:,12]
    valid_pct = 100 * np.isfinite(pb[:n]).sum() / n if n > 0 else 0

    lag_r = bc.best_lag(pb[:n], gt[:n], max_lag=MAX_LAG)
    lag   = lag_r[0] if isinstance(lag_r, tuple) else lag_r
    p_al, t_al = bc.overlap(pb[:n], gt[:n], lag)

    mask_a = np.isfinite(pa[:n]) & np.isfinite(gt[:n])
    mask_b = np.isfinite(p_al)   & np.isfinite(t_al)

    mae_a  = np.mean(np.abs(pa[:n][mask_a]-gt[:n][mask_a])) if mask_a.sum() else np.nan
    mae_b  = np.mean(np.abs(p_al[mask_b]-t_al[mask_b]))     if mask_b.sum() else np.nan
    rmse_b = np.sqrt(np.mean((p_al[mask_b]-t_al[mask_b])**2)) if mask_b.sum() else np.nan
    bias_b = np.mean(p_al[mask_b]-t_al[mask_b])               if mask_b.sum() else np.nan
    r_b    = np.corrcoef(p_al[mask_b],t_al[mask_b])[0,1]      if mask_b.sum()>2 else np.nan
    gt20_b = 100*(np.abs(p_al[mask_b]-t_al[mask_b])>20).mean() if mask_b.sum() else np.nan

    print(f"Ex{ex} {rec['rec_id']}: "
          f"valid={valid_pct:.0f}%  "
          f"MAE_A={mae_a:.1f}°  MAE_B={mae_b:.1f}°  "
          f"r={r_b:.3f}")

    results_side.append(dict(
        rec=rec, ex=ex, t=t, n=n, fps=fps,
        gt=gt, pa=pa, pb=pb, p_al=p_al, t_al=t_al,
        mae_a=mae_a, mae_b=mae_b, rmse_b=rmse_b,
        bias_b=bias_b, r_b=r_b, gt20_b=gt20_b,
        valid_pct=valid_pct
    ))

# ── PLOT: time series ─────────────────────────────────────────
n_plots = len(results_side)
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 5*n_plots))
if n_plots == 1: axes = [axes]

for ax, res in zip(axes, results_side):
    t  = res['t'];  n = res['n']
    gt = res['gt']; pa= res['pa']; pb= res['pb']

    ax.fill_between(t[:n], gt[:n]-10, gt[:n]+10,
                    alpha=0.10, color='black',
                    label='±10° tolerance')
    ax.plot(t[:n], gt[:n], color='black', lw=2.5, zorder=4,
            label='OptiTrack — Ground Truth (Left arm)')

    if np.isfinite(pa[:n]).sum() > 10:
        ax.plot(t[:n], pa[:n], color='#E74C3C', lw=1.5,
                ls='--', alpha=0.85,
                label=f'Variant A — 2D  MAE={res["mae_a"]:.1f}°')
    else:
        ax.text(0.05, 0.65,
                'Variant A (2D) — insufficient valid frames',
                transform=ax.transAxes, color='#E74C3C', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='#FADBD8', alpha=0.8))

    if np.isfinite(pb[:n]).sum() > 10:
        ax.plot(t[:n], pb[:n], color='#27AE60', lw=1.8, alpha=0.9,
                label=f'Variant B — 3D  MAE={res["mae_b"]:.1f}°  '
                      f'r={res["r_b"]:.3f}')
    else:
        ax.text(0.05, 0.55,
                'Variant B (3D) — insufficient valid frames',
                transform=ax.transAxes, color='#27AE60', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='#D5F5E3', alpha=0.8))

    nan_b = np.isnan(pb[:n])
    if nan_b.any():
        ax.fill_between(t[:n], 0, 185, where=nan_b,
                        alpha=0.15, color='grey',
                        label=f'Lost tracking ({100*nan_b.sum()/n:.0f}%)')

    ax2 = ax.twinx()
    if np.isfinite(pb[:n]).sum() > 10:
        ax2.plot(t[:n], np.abs(pb[:n]-gt[:n]),
                 color='#F39C12', lw=0.8, alpha=0.5)
    ax2.set_ylabel('|Error B| (deg)', color='#F39C12', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='#F39C12')
    ax2.set_ylim(0, 60)

    ax.set_title(
        f"{ex_labels[res['ex']]}  —  {res['rec']['rec_id']}  "
        f"—  Side view (cam18)  —  Left arm  —  valid={res['valid_pct']:.0f}%",
        fontsize=11, fontweight='bold')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Elbow angle (degrees)')
    ax.set_ylim(0, 190)
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.25)

plt.suptitle(
    'Side View (cam18) — Left Arm: Predicted vs OptiTrack\n'
    'Black = Ground Truth  |  Red dashed = 2D (A)  |  '
    'Green = 3D (B)  |  Orange = |Error|',
    fontsize=12, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS,'sideview_left_arm.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: sideview_left_arm.png")

# ── scatter plots ─────────────────────────────────────────────
fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 5))
if n_plots == 1: axes = [axes]

for ax, res in zip(axes, results_side):
    mask = np.isfinite(res['p_al']) & np.isfinite(res['t_al'])
    if mask.sum() < 5:
        ax.set_title(f"{ex_labels[res['ex']]}\nNo data")
        continue
    pp = res['p_al'][mask]
    tt = res['t_al'][mask]
    ax.scatter(tt, pp, alpha=0.08, s=3,
               color='#27AE60', rasterized=True)
    lims = [max(0,  min(tt.min(), pp.min())-5),
            min(185, max(tt.max(), pp.max())+5)]
    ax.plot(lims, lims, 'k--', lw=2, label='Perfect agreement')
    ax.plot(lims, [l+15 for l in lims], color='orange',
            lw=1.2, ls=':', alpha=0.8)
    ax.plot(lims, [l-15 for l in lims], color='orange',
            lw=1.2, ls=':', alpha=0.8, label='±15°')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('OptiTrack angle (degrees)', fontsize=9)
    ax.set_ylabel('Variant B predicted (degrees)', fontsize=9)
    ax.set_title(
        f"{ex_labels[res['ex']]}\nr={res['r_b']:.3f}  MAE={res['mae_b']:.1f}°",
        fontweight='bold', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)

plt.suptitle(
    'Scatter: Variant B (3D) vs OptiTrack — Side View (Left Arm)\n'
    'Each dot = one frame  |  Black dashed = perfect  |  Orange = ±15°',
    fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS,'sideview_scatter.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: sideview_scatter.png")

# ── front vs side comparison table ───────────────────────────
print("\n" + "="*72)
print("KEY RESULT: FRONT VIEW vs SIDE VIEW — MAE Comparison")
print("="*72)
print(f"{'Exercise':<22} {'Front_A':>8} {'Front_B':>8} "
      f"{'Side_A':>8} {'Side_B':>8} {'3D_improvement':>15}")
print("-"*72)
front_by_ex = {res['ex']: res for res in results}
for res_s in results_side:
    ex    = res_s['ex']
    front = front_by_ex.get(ex)
    maf   = front['mae_a'] if front and not np.isnan(front['mae_a']) else np.nan
    mbf   = front['mae_b'] if front and not np.isnan(front['mae_b']) else np.nan
    mas   = res_s['mae_a']
    mbs   = res_s['mae_b']
    imp   = mas - mbs if not np.isnan(mas) and not np.isnan(mbs) else np.nan
    print(f"{ex_labels[ex]:<22} "
          f"{maf:>8.1f}° {mbf:>8.1f}° "
          f"{mas:>8.1f}° {mbs:>8.1f}° "
          f"{imp:>+15.1f}°")

print()
print("3D_improvement = Side MAE_A - Side MAE_B")
print("Positive = Variant B (3D) is better than Variant A (2D) from side view")
print("This proves 3D world landmarks help most when camera angle is challenging")

In [ ]:
# check exactly which recording is in best_cam18 for each exercise
for ex, c in sorted(best_cam18.items()):
    cp = c['cp']
    d  = np.load(cp)
    pb_raw = elbow_angle_3d(
        mask_world_like(d['world'],
                        mask_low_visibility(d['lm'], threshold=THRESHOLD)),
        side='left')
    pb = fill_1d(pb_raw)
    print(f"Ex{ex}: {c['rec']['rec_id']}")
    print(f"       cache file : {os.path.basename(cp)}")
    print(f"       pb finite  : {np.isfinite(pb).sum()}/{len(pb)}")
    print()

In [ ]:
for c in sorted(best_cam18.values(), key=lambda x: x['rec']['exercise']):
    rec = c['rec']
    cp  = c['cp']
    print(f"rec_id    : {rec['rec_id']}")
    print(f"cache file: {os.path.basename(cp)}")
    print(f"cp exists : {os.path.exists(cp)}")
    print()

In [ ]:
# ============================================================
# SIDE VIEW FIXED v3 — use cp directly from best_cam18
# ============================================================
import numpy as np, matplotlib.pyplot as plt, os

import angles as angles_2d
from angles_world import elbow_angle_3d, mask_world_like
import benchmark_common as bc
from benchmark_common import elbow_angle_3d_ordered
from benchmark_common import butterworth_filter
from visibility import mask_low_visibility

def fill_1d(arr, max_gap=15):
    arr = np.array(arr, dtype=float)
    if np.isnan(arr).all() or not np.isnan(arr).any():
        return arr
    idx  = np.arange(len(arr))
    good = ~np.isnan(arr)
    filled = arr.copy()
    in_gap, gap_start = False, 0
    for i in range(len(arr)):
        if np.isnan(arr[i]) and not in_gap:
            in_gap, gap_start = True, i
        elif not np.isnan(arr[i]) and in_gap:
            in_gap = False
            if i - gap_start <= max_gap:
                filled[gap_start:i] = np.interp(
                    idx[gap_start:i], idx[good], arr[good])
    return filled

def safe_smooth(arr, fps, cutoff=6.0):
    if np.isfinite(arr).sum() < 15:
        return arr
    try:
        smoothed = butterworth_filter(arr, fs=fps, cutoff=cutoff)
        if np.isfinite(smoothed).sum() == 0:
            return arr
        return smoothed
    except Exception:
        return arr

ex_labels = {1:'Elbow Flexion (Ex1)',
             2:'Arm VW (Ex2)',
             3:'Table Push-ups (Ex3)'}

SIDE_CAM18 = 'left'
ARMS_LEFT  = ('7: LeftArm', '8: LeftForeArm', '9: LeftHand')

# use cp DIRECTLY from best_cam18 — no path rebuilding
sample_cam18 = [
    (c['rec'], c['cp'], c['rec']['exercise'])
    for c in sorted(best_cam18.values(),
                    key=lambda x: x['rec']['exercise'])
]

print(f"Plotting {len(sample_cam18)} side-view recordings")
for rec, cp, ex in sample_cam18:
    print(f"  Ex{ex}: {rec['rec_id']}  cp={os.path.basename(cp)}  "
          f"exists={os.path.exists(cp)}")

results_side = []
for rec, cp, ex in sample_cam18:
    print(f"\nProcessing Ex{ex} {rec['rec_id']}...")

    d      = np.load(cp)
    lm     = d['lm'];  world = d['world']
    size   = tuple(d['size']);  fps = float(d['fps'])
    npz_f0 = int(d['f0'])

    truth_arr = bc.load_truth_array(rec['truth_path'], JOINT_NAMES)
    gt = truth_angle_for(rec, truth_arr, world, npz_f0, fps, ARMS_LEFT)

    print(f"  gt finite: {np.isfinite(gt).sum()}/{len(gt)}")

    lm_m  = mask_low_visibility(lm,    threshold=THRESHOLD)
    wld_m = mask_world_like(world, lm_m)

    pa_raw = angles_2d.elbow_angle(lm_m, side=SIDE_CAM18, size=size)
    pa     = fill_1d(pa_raw)
    if np.isfinite(pa).sum() > 15:
        pa = butterworth_filter(pa, fs=fps, cutoff=6.0)
    print(f"  pa finite: {np.isfinite(pa).sum()}/{len(pa)}")

    pb_raw = elbow_angle_3d(wld_m, side=SIDE_CAM18)
    pb     = fill_1d(pb_raw)
    if np.isfinite(pb).sum() > 15:
        pb = butterworth_filter(pb, fs=fps, cutoff=6.0)
    print(f"  pb finite: {np.isfinite(pb).sum()}/{len(pb)}")

    n = min(len(pa), len(pb), len(gt))
    t = np.arange(n) / fps

    # valid% from pb — computed AFTER pb is ready
    valid_pct = 100 * np.isfinite(pb[:n]).sum() / n if n > 0 else 0
    print(f"  n={n}  valid_pct={valid_pct:.0f}%")

    lag_r = bc.best_lag(pb[:n], gt[:n], max_lag=MAX_LAG)
    lag   = lag_r[0] if isinstance(lag_r, tuple) else lag_r
    p_al, t_al = bc.overlap(pb[:n], gt[:n], lag)

    mask_a = np.isfinite(pa[:n]) & np.isfinite(gt[:n])
    mask_b = np.isfinite(p_al)   & np.isfinite(t_al)

    print(f"  mask_a sum={mask_a.sum()}  mask_b sum={mask_b.sum()}")

    mae_a  = np.mean(np.abs(pa[:n][mask_a]-gt[:n][mask_a])) if mask_a.sum() else np.nan
    mae_b  = np.mean(np.abs(p_al[mask_b]-t_al[mask_b]))     if mask_b.sum() else np.nan
    rmse_b = np.sqrt(np.mean((p_al[mask_b]-t_al[mask_b])**2)) if mask_b.sum() else np.nan
    bias_b = np.mean(p_al[mask_b]-t_al[mask_b])               if mask_b.sum() else np.nan
    r_b    = np.corrcoef(p_al[mask_b],t_al[mask_b])[0,1]      if mask_b.sum()>2 else np.nan
    gt20_b = 100*(np.abs(p_al[mask_b]-t_al[mask_b])>20).mean() if mask_b.sum() else np.nan

    print(f"  MAE_A={mae_a:.1f}  MAE_B={mae_b:.1f}  r={r_b:.3f}")

    results_side.append(dict(
        rec=rec, ex=ex, t=t, n=n, fps=fps,
        gt=gt, pa=pa, pb=pb, p_al=p_al, t_al=t_al,
        mae_a=mae_a, mae_b=mae_b, rmse_b=rmse_b,
        bias_b=bias_b, r_b=r_b, gt20_b=gt20_b,
        valid_pct=valid_pct
    ))

# ── plots ─────────────────────────────────────────────────────
n_plots = len(results_side)
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 5*n_plots))
if n_plots == 1: axes = [axes]

for ax, res in zip(axes, results_side):
    t  = res['t'];  n = res['n']
    gt = res['gt']; pa= res['pa']; pb= res['pb']

    ax.fill_between(t[:n], gt[:n]-10, gt[:n]+10,
                    alpha=0.10, color='black', label='±10° tolerance')
    ax.plot(t[:n], gt[:n], color='black', lw=2.5, zorder=4,
            label='OptiTrack — Ground Truth (Left arm)')

    if np.isfinite(pa[:n]).sum() > 10:
        ax.plot(t[:n], pa[:n], color='#E74C3C', lw=1.5,
                ls='--', alpha=0.85,
                label=f'Variant A — 2D  MAE={res["mae_a"]:.1f}°')
    else:
        ax.text(0.05, 0.65,
                'Variant A (2D) — too few valid frames from side',
                transform=ax.transAxes, color='#E74C3C', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='#FADBD8', alpha=0.8))

    if np.isfinite(pb[:n]).sum() > 10:
        ax.plot(t[:n], pb[:n], color='#27AE60', lw=1.8, alpha=0.9,
                label=f'Variant B — 3D  MAE={res["mae_b"]:.1f}°  '
                      f'r={res["r_b"]:.3f}')
    else:
        ax.text(0.05, 0.50,
                'Variant B (3D) — too few valid frames from side',
                transform=ax.transAxes, color='#27AE60', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='#D5F5E3', alpha=0.8))

    nan_b = np.isnan(pb[:n])
    if nan_b.any():
        ax.fill_between(t[:n], 0, 185, where=nan_b,
                        alpha=0.15, color='grey',
                        label=f'Lost tracking ({100*nan_b.sum()/n:.0f}%)')

    ax2 = ax.twinx()
    if np.isfinite(pb[:n]).sum() > 10:
        ax2.plot(t[:n], np.abs(pb[:n]-gt[:n]),
                 color='#F39C12', lw=0.8, alpha=0.5)
    ax2.set_ylabel('|Error B| (deg)', color='#F39C12', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='#F39C12')
    ax2.set_ylim(0, 60)

    ax.set_title(
        f"{ex_labels[res['ex']]}  —  {res['rec']['rec_id']}  "
        f"—  Side view (cam18)  —  Left arm  —  valid={res['valid_pct']:.0f}%",
        fontsize=11, fontweight='bold')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Elbow angle (degrees)')
    ax.set_ylim(0, 190)
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.25)

plt.suptitle(
    'Side View (cam18) — Left Arm: Predicted vs OptiTrack\n'
    'Black = Ground Truth  |  Red dashed = 2D (A)  |  '
    'Green = 3D (B)  |  Orange = |Error|',
    fontsize=12, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS,'sideview_left_arm.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: sideview_left_arm.png")

# ── front vs side summary table ───────────────────────────────
print("\n" + "="*72)
print("KEY RESULT: FRONT VIEW vs SIDE VIEW")
print("="*72)
print(f"{'Exercise':<22} {'Front_A':>8} {'Front_B':>8} "
      f"{'Side_A':>8} {'Side_B':>8} {'B_improvement':>14}")
print("-"*72)
front_by_ex = {res['ex']: res for res in results}
for res_s in results_side:
    ex  = res_s['ex']
    f   = front_by_ex.get(ex)
    maf = f['mae_a'] if f and not np.isnan(f['mae_a']) else np.nan
    mbf = f['mae_b'] if f and not np.isnan(f['mae_b']) else np.nan
    mas = res_s['mae_a']
    mbs = res_s['mae_b']
    imp = mas - mbs if not np.isnan(mas) and not np.isnan(mbs) else np.nan
    print(f"{ex_labels[ex]:<22} "
          f"{maf:>8.1f}° {mbf:>8.1f}° "
          f"{mas:>8.1f}° {mbs:>8.1f}° "
          f"{imp:>+14.1f}°")

print()
print("B_improvement = Side MAE_A - Side MAE_B")
print("Positive = Variant B (3D) beats Variant A (2D) from side view")

In [ ]:
# check why Ex1 and Ex3 cam18 landmarks are all dropped
for ex in [1, 3]:
    c  = best_cam18[ex]
    d  = np.load(c['cp'])
    lm = d['lm']

    print(f"=== Ex{ex} {c['rec']['rec_id']} ===")
    # left arm joints: 11=shoulder, 13=elbow, 15=wrist
    for joint, name in [(11,'L.shoulder'), (13,'L.elbow'), (15,'L.wrist')]:
        vis = lm[:, joint, 3]
        print(f"  {name}: mean={np.nanmean(vis):.3f}  "
              f"min={np.nanmin(vis):.3f}  "
              f"above_0.7={100*(vis>=0.7).mean():.1f}%")

    # check what mask_low_visibility actually does
    from visibility import mask_low_visibility
    lm_m = mask_low_visibility(lm, threshold=THRESHOLD)
    print(f"  After masking:")
    for joint, name in [(11,'L.shoulder'), (13,'L.elbow'), (15,'L.wrist')]:
        print(f"    {name}: finite={np.isfinite(lm_m[:,joint,0]).sum()}/{len(lm)}")

In [ ]:
from angles_world import elbow_angle_3d, mask_world_like
from visibility import mask_low_visibility

for ex in [1, 3]:
    c  = best_cam18[ex]
    d  = np.load(c['cp'])
    lm    = d['lm']
    world = d['world']

    print(f"=== Ex{ex} {c['rec']['rec_id']} ===")

    # check world landmarks directly
    for joint, name in [(11,'L.shoulder'), (13,'L.elbow'), (15,'L.wrist')]:
        print(f"  world {name}: "
              f"finite={np.isfinite(world[:,joint,0]).sum()}/{len(world)}  "
              f"sample={world[:3,joint,:3]}")

    lm_m  = mask_low_visibility(lm, threshold=THRESHOLD)
    wld_m = mask_world_like(world, lm_m)

    # check world after masking
    print(f"  After mask_world_like:")
    for joint, name in [(11,'L.shoulder'), (13,'L.elbow'), (15,'L.wrist')]:
        print(f"    {name}: finite={np.isfinite(wld_m[:,joint,0]).sum()}/{len(world)}")

    # try angle directly
    ang = elbow_angle_3d(wld_m, side='left')
    print(f"  elbow_angle_3d left: finite={np.isfinite(ang).sum()}/{len(ang)}")
    print(f"  sample angles: {ang[:5]}")
    print()

In [ ]:
from benchmark_common import butterworth_filter
import numpy as np

for ex in [1, 2, 3]:
    c  = best_cam18[ex]
    d  = np.load(c['cp'])
    from angles_world import elbow_angle_3d, mask_world_like
    from visibility import mask_low_visibility
    lm_m  = mask_low_visibility(d['lm'], threshold=THRESHOLD)
    wld_m = mask_world_like(d['world'], lm_m)
    fps   = float(d['fps'])

    pb_raw = elbow_angle_3d(wld_m, side='left')
    pb_f   = fill_1d(pb_raw)
    print(f"Ex{ex}: fps={fps:.1f}  pb_raw finite={np.isfinite(pb_raw).sum()}  "
          f"pb_fill finite={np.isfinite(pb_f).sum()}")

    try:
        pb_smooth = butterworth_filter(pb_f, fs=fps, cutoff=6.0)
        print(f"  butterworth OK: finite={np.isfinite(pb_smooth).sum()}")
    except Exception as e:
        print(f"  butterworth FAILED: {e}")


In [40]:
import pandas as pd, glob, os
df = pd.concat([pd.read_csv(f) for f in glob.glob(os.path.join(RESULTS,'variant_*.csv'))],
               ignore_index=True)
print('rows:', len(df))
print(df.variant.value_counts())

rows: 280
variant
B_3D_World         56
A_2D_Base          56
D_2D_Calibrated    56
D_3D_Calibrated    56
D_2D_PerView       56
Name: count, dtype: int64


In [41]:
def rom(a):
    a = np.asarray(a, float)
    return (np.nanpercentile(a,95) - np.nanpercentile(a,5)
            if np.isfinite(a).any() else np.nan)

import collections
acc = collections.defaultdict(list)
for item in aligned_cache:
    me = item['meta']
    pool = [x for x in aligned_cache if x['meta']['person'] != me['person']]
    same = [x for x in pool if x['meta']['view'] == me['view']] or pool
    t = rom(item['truth_3d'])
    if not np.isfinite(t) or t <= 0:
        continue
    acc['raw'].append(rom(item['pred_2d']) / t)
    for train, name in ((pool,'global'), (same,'perview')):
        p_cat = np.concatenate([x['pred_2d']  for x in train])
        t_cat = np.concatenate([x['truth_3d'] for x in train])
        v = np.isfinite(p_cat) & np.isfinite(t_cat)
        try:
            prm = fit_curve(p_cat[v], t_cat[v], min_count=15)
            acc[name].append(rom(apply_calibration(item['pred_2d'], prm)) / t)
        except ValueError:
            pass

for k in ('raw','global','perview'):
    v = [x for x in acc[k] if np.isfinite(x)]
    if v:
        print(f'{k:8} ROM/truth  median {np.median(v):.3f}   '
              f'spread {min(v):.2f}-{max(v):.2f}   n={len(v)}')

raw      ROM/truth  median 1.075   spread 0.29-1.73   n=56
global   ROM/truth  median 0.863   spread 0.33-1.16   n=56
perview  ROM/truth  median 0.945   spread 0.38-1.20   n=56


In [42]:
print('--- overall ---')
print(df.groupby('variant')[['mae','rmse','bias','r','share_gt20']].median().round(2))
print()
A = df.copy()
A['subj'] = A['view'].str.replace('/cam18', '', regex=False)
print('--- by camera x subject orientation ---')
print(A[A.variant.isin(['A_2D_Base','D_2D_Calibrated','D_2D_PerView'])]
        .groupby(['variant','cam','subj'])[['mae','bias']]
        .agg(['count','median']).round(2))

--- overall ---
                   mae   rmse   bias     r  share_gt20
variant                                               
A_2D_Base        16.12  23.83   7.99  0.91       30.40
B_3D_World       21.93  26.56  14.90  0.89       48.69
D_2D_Calibrated  14.19  18.51   3.79  0.93       20.80
D_2D_PerView     13.33  17.85   2.46  0.93       19.28
D_3D_Calibrated  15.02  19.37   4.46  0.91       25.85

--- by camera x subject orientation ---
                                     mae         bias       
                                   count median count median
variant         cam   subj                                  
A_2D_Base       cam17 Front           18  12.00    18  -1.31
                      Half-profile     5  17.70     5   2.24
                      Profile         10  16.77    10  13.18
                cam18 Front            6  24.70     6   8.49
                      Half-profile     4  13.76     4  -3.81
                      Profile         13  20.23    13  19.09
D_2D_Cali